In [1]:
!pip install duckdb

In [2]:
import duckdb

In [3]:
con = duckdb.connect()

In [4]:
con.sql("Select 1").show()

┌───────┐
│   1   │
│ int32 │
├───────┤
│     1 │
└───────┘



In [5]:
import pandas as pd

**Load the products sheet from Excel into a Pandas DataFrame.**

What This Code Does
file_path

Stores the location of your uploaded Excel file.

pd.read_excel(...)

Reads the "products" sheet from the Excel file.

products

Becomes a Pandas DataFrame containing:

product_code

product_type

products.head()

Shows first 5 rows so we can:

Inspect structure

Verify columns

Detect data issues early

🔹 Why We Are Doing This

Before loading into SQL:

We must inspect structure.

Confirm column names.

Check if any cleaning is required.

Validate dataset integrity.

This is called staging inspection in Data Engineering.

In [6]:
file_path = "/content/data.xlsx"
products = pd.read_excel(file_path, sheet_name = 'products')
products.head()

,product_code,product_type
0,Prod001,Own Brand
1,Prod002,Own Brand
2,Prod003,Own Brand
3,Prod004,Own Brand
4,Prod005,Own Brand


**Register DataFrame in DuckDB**
🔹 Heading

Make the products DataFrame available inside DuckDB as a queryable object.

In [7]:
con.register("products_df",products)

Treat this Pandas DataFrame as a temporary SQL table named products_df.”
Important:

It does NOT create a permanent SQL table yet.

It just exposes it for SQL queries.

It lives in memory.

In [8]:
con.sql("SELECT * FROM products_df LIMIT 5").show()

┌──────────────┬──────────────┐
│ product_code │ product_type │
│   varchar    │   varchar    │
├──────────────┼──────────────┤
│ Prod001      │ Own Brand    │
│ Prod002      │ Own Brand    │
│ Prod003      │ Own Brand    │
│ Prod004      │ Own Brand    │
│ Prod005      │ Own Brand    │
└──────────────┴──────────────┘



**Create a physical table inside DuckDB from the registered DataFrame. **

In [9]:
con.sql(

    """
    CREATE TABLE products AS
    SELECT *
    FROM products_df
    """
)

You should see the row count. For verifying

In [10]:
con.sql(
    """
    SELECT COUNT(*)
    FROM products
    """
).show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          279 │
└──────────────┘



**Load the customers sheet into a Pandas DataFrame.**

What This Code Does

Reads the customers sheet from Excel.

Stores it in a DataFrame named customers.

Displays first 5 rows for inspection.

You should see columns like:

customer_code

customer_name

customer_type

In [11]:
customers = pd.read_excel(file_path, sheet_name="customers")

customers.head()

,customer_code,customer_name,customer_type
0,Cus001,Surge Stores,Brick & Mortar
1,Cus002,Nomad Stores,Brick & Mortar
2,Cus003,Excel Stores,Brick & Mortar
3,Cus004,Surface Stores,Brick & Mortar
4,Cus005,Premium Stores,Brick & Mortar


**Make the customers DataFrame queryable inside DuckDB.**





In [12]:
con.register("customers_df",customers)
con.sql(
    """
    SELECT *
    FROM customers_df
    LIMIT 5
    """
).show()

┌───────────────┬────────────────┬────────────────┐
│ customer_code │ customer_name  │ customer_type  │
│    varchar    │    varchar     │    varchar     │
├───────────────┼────────────────┼────────────────┤
│ Cus001        │ Surge Stores   │ Brick & Mortar │
│ Cus002        │ Nomad Stores   │ Brick & Mortar │
│ Cus003        │ Excel Stores   │ Brick & Mortar │
│ Cus004        │ Surface Stores │ Brick & Mortar │
│ Cus005        │ Premium Stores │ Brick & Mortar │
└───────────────┴────────────────┴────────────────┘



**Create a physical SQL table from the staged DataFrame.**

In [13]:
con.sql(
    """
    CREATE TABLE customers AS
    SELECT *
    FROM customers_df
    """
)

**You should see the row count. For verifying**

In [14]:
con.sql("SELECT COUNT(*) FROM customers").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│           38 │
└──────────────┘



In [15]:
con.sql("SHOW TABLES").show()

┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ customers    │
│ customers_df │
│ products     │
│ products_df  │
└──────────────┘



**Load the markets sheet into Pandas for inspection.**

In [16]:
markets = pd.read_excel(file_path, sheet_name="markets")
markets.head()

,market_code,markets_name,zone
0,Mark001,Chennai,South
1,Mark002,Mumbai,Central
2,Mark003,Ahmedabad,North
3,Mark004,Delhi NCR,North
4,Mark005,Kanpur,North


**Make the markets DataFrame queryable inside DuckDB.**


In [17]:
con.register("markets_df",markets)
con.sql(
    """
    SELECT *
    FROM markets_df
    LIMIT 5
    """
)

┌─────────────┬──────────────┬─────────┐
│ market_code │ markets_name │  zone   │
│   varchar   │   varchar    │ varchar │
├─────────────┼──────────────┼─────────┤
│ Mark001     │ Chennai      │ South   │
│ Mark002     │ Mumbai       │ Central │
│ Mark003     │ Ahmedabad    │ North   │
│ Mark004     │ Delhi NCR    │ North   │
│ Mark005     │ Kanpur       │ North   │
└─────────────┴──────────────┴─────────┘

**Create a physical SQL table from the staged DataFrame.**

In [18]:
con.sql(
    """
    CREATE TABLE markets AS
    SELECT *
    FROM markets_df
    """
)

**Checking the Table**

In [19]:
con.sql(
    """
    SELECT * FROM markets
    LIMIT 5
    """
)

┌─────────────┬──────────────┬─────────┐
│ market_code │ markets_name │  zone   │
│   varchar   │   varchar    │ varchar │
├─────────────┼──────────────┼─────────┤
│ Mark001     │ Chennai      │ South   │
│ Mark002     │ Mumbai       │ Central │
│ Mark003     │ Ahmedabad    │ North   │
│ Mark004     │ Delhi NCR    │ North   │
│ Mark005     │ Kanpur       │ North   │
└─────────────┴──────────────┴─────────┘

In [20]:
con.sql("SHOW TABLES").show()

┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ customers    │
│ customers_df │
│ markets      │
│ markets_df   │
│ products     │
│ products_df  │
└──────────────┘



**Load the markets sheet into Pandas for inspection.**

In [21]:
dim_date = pd.read_excel(file_path,sheet_name="date")
dim_date.head()

,date,cy-date,year,month_name,date_yy_mmm
0,2017-06-01,2017-06-01,2017,June,2022-06-17
1,2017-06-02,2017-06-01,2017,June,2022-06-17
2,2017-06-03,2017-06-01,2017,June,2022-06-17
3,2017-06-04,2017-06-01,2017,June,2022-06-17
4,2017-06-05,2017-06-01,2017,June,2022-06-17


**Make the date DataFrame queryable inside DuckDB.**


In [22]:
con.register("dim_date",dim_date)
con.sql(
    """
    SELECT * FROM dim_date
    LIMIT 5
    """
)

┌─────────────────────┬─────────────────────┬───────┬────────────┬─────────────────────┐
│        date         │       cy-date       │ year  │ month_name │     date_yy_mmm     │
│    timestamp_ns     │    timestamp_ns     │ int64 │  varchar   │    timestamp_ns     │
├─────────────────────┼─────────────────────┼───────┼────────────┼─────────────────────┤
│ 2017-06-01 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-02 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-03 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-04 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-05 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
└─────────────────────┴─────────────────────┴───────┴────────────┴─────────────────────┘

**Create a physical SQL table from the staged DataFrame.**

In [23]:
con.sql(
    """
    CREATE TABLE dim_date AS
    SELECT * FROM dim_date
    LIMIT 5
    """
)

In [24]:
con.sql(
    """
    SELECT * FROM dim_date
    LIMIT 5
    """
)

┌─────────────────────┬─────────────────────┬───────┬────────────┬─────────────────────┐
│        date         │       cy-date       │ year  │ month_name │     date_yy_mmm     │
│    timestamp_ns     │    timestamp_ns     │ int64 │  varchar   │    timestamp_ns     │
├─────────────────────┼─────────────────────┼───────┼────────────┼─────────────────────┤
│ 2017-06-01 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-02 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-03 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-04 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-05 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
└─────────────────────┴─────────────────────┴───────┴────────────┴─────────────────────┘

In [25]:
con.sql("SHOW TABLES").show()

┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ customers    │
│ customers_df │
│ dim_date     │
│ dim_date     │
│ markets      │
│ markets_df   │
│ products     │
│ products_df  │
└──────────────┘



**Load the markets sheet into Pandas for inspection.**

In [26]:
transactions = pd.read_excel(file_path,sheet_name="transactions")
transactions.head()

,product_code,customer_code,market_code,order_date,sales_qty,sales_amount,currency
0,Prod001,Cus001,Mark001,2017-10-10,100,41241,INR
1,Prod001,Cus002,Mark002,2018-05-08,3,-1,INR
2,Prod002,Cus003,Mark003,2018-04-06,1,875,INR
3,Prod002,Cus003,Mark003,2018-04-11,1,583,INR
4,Prod002,Cus004,Mark003,2018-06-18,6,7176,INR


**Make the Transactions DataFrame queryable inside DuckDB.**


In [27]:
con.register("transactions_df", transactions)
con.sql(
    """
    SELECT * FROM transactions
    LIMIT 5
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod001      │ Cus001        │ Mark001     │ 2017-10-10 00:00:00 │       100 │        41241 │ INR      │
│ Prod001      │ Cus002        │ Mark002     │ 2018-05-08 00:00:00 │         3 │           -1 │ INR      │
│ Prod002      │ Cus003        │ Mark003     │ 2018-04-06 00:00:00 │         1 │          875 │ INR      │
│ Prod002      │ Cus003        │ Mark003     │ 2018-04-11 00:00:00 │         1 │          583 │ INR      │
│ Prod002      │ Cus004        │ Mark003     │ 2018-06-18 00:00:00 │         6 │         7176 │ INR      │
└──────────────┴───────────────┴─────

**Create a physical SQL table from the staged DataFrame.**

In [28]:
con.sql(
    """
    CREATE TABLE transactions AS
    SELECT * FROM transactions_df
    """
)

In [29]:
con.sql(
    """
    SELECT * FROM transactions
    LIMIT 5
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod001      │ Cus001        │ Mark001     │ 2017-10-10 00:00:00 │       100 │        41241 │ INR      │
│ Prod001      │ Cus002        │ Mark002     │ 2018-05-08 00:00:00 │         3 │           -1 │ INR      │
│ Prod002      │ Cus003        │ Mark003     │ 2018-04-06 00:00:00 │         1 │          875 │ INR      │
│ Prod002      │ Cus003        │ Mark003     │ 2018-04-11 00:00:00 │         1 │          583 │ INR      │
│ Prod002      │ Cus004        │ Mark003     │ 2018-06-18 00:00:00 │         6 │         7176 │ INR      │
└──────────────┴───────────────┴─────

In [30]:
con.sql("SHOW TABLES").show()

┌─────────────────┐
│      name       │
│     varchar     │
├─────────────────┤
│ customers       │
│ customers_df    │
│ dim_date        │
│ dim_date        │
│ markets         │
│ markets_df      │
│ products        │
│ products_df     │
│ transactions    │
│ transactions_df │
├─────────────────┤
│     10 rows     │
└─────────────────┘



**Check that all tables are created in DuckDB**
**Validate Row Counts**

In [31]:
con.sql("""
SELECT 'products' AS table_name, COUNT(*) FROM products
UNION ALL
SELECT 'customers', COUNT(*) FROM customers
UNION ALL
SELECT 'markets', COUNT(*) FROM markets
UNION ALL
SELECT 'dim_date', COUNT(*) FROM dim_date
UNION ALL
SELECT 'transactions', COUNT(*) FROM transactions
""").show()

┌──────────────┬──────────────┐
│  table_name  │ count_star() │
│   varchar    │    int64     │
├──────────────┼──────────────┤
│ products     │          279 │
│ customers    │           38 │
│ markets      │           17 │
│ dim_date     │         1126 │
│ transactions │       150283 │
└──────────────┴──────────────┘



**MODULE 1 — Filtering (WHERE Clause)**

🔹 Step 2 — Basic Filtering

Filters rows where sales_amount is greater than 10,000.

Returns only high-value transactions.

Shows how WHERE reduces dataset size before any aggregation.

In [32]:
con.sql(
    """
    SELECT *
    FROM transactions
    WHERE sales_amount >= 10000
    LIMIT 5
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod001      │ Cus001        │ Mark001     │ 2017-10-10 00:00:00 │       100 │        41241 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-23 00:00:00 │        39 │        21412 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-27 00:00:00 │        35 │        19213 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-28 00:00:00 │       310 │       170185 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-29 00:00:00 │       184 │       101194 │ INR      │
└──────────────┴───────────────┴─────

**MODULE 1 — Multiple Conditions (AND / OR)**

🔹 Step 3 — Filter with AND

What This Does

First filters sales_amount > 10000

Then further filters only rows where currency = 'INR'

Both conditions must be TRUE.

In [33]:
con.sql(
    """
    SELECT *
    FROM transactions
    WHERE sales_amount >= 10000
      AND currency = 'INR'
    LIMIT 5
    """
)


┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod001      │ Cus001        │ Mark001     │ 2017-10-10 00:00:00 │       100 │        41241 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-23 00:00:00 │        39 │        21412 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-27 00:00:00 │        35 │        19213 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-28 00:00:00 │       310 │       170185 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-29 00:00:00 │       184 │       101194 │ INR      │
└──────────────┴───────────────┴─────

In [34]:
con.sql(
    """
    SELECT *
    FROM transactions
    WHERE sales_amount >= 10000
      OR sales_qty > 500
    LIMIT 5
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod001      │ Cus001        │ Mark001     │ 2017-10-10 00:00:00 │       100 │        41241 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-23 00:00:00 │        39 │        21412 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-27 00:00:00 │        35 │        19213 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-28 00:00:00 │       310 │       170185 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-29 00:00:00 │       184 │       101194 │ INR      │
└──────────────┴───────────────┴─────

**MODULE 1 — DISTINCT & Data Exploration**

🔹 Step 4 — Find Unique Values

In [35]:
con.sql(
    """
    SELECT DISTINCT currency
    FROM transactions
    """
)

┌──────────┐
│ currency │
│ varchar  │
├──────────┤
│ INR      │
│ USD      │
└──────────┘

In [36]:
con.sql(
    """
    SELECT DISTINCT zone
    FROM markets
    """
)

┌─────────┐
│  zone   │
│ varchar │
├─────────┤
│ South   │
│ Central │
│ North   │
│ NULL    │
└─────────┘

**MODULE 1 — ORDER BY (Sorting Data)**

🔹 Step 6 — Find Highest Sales Transactions

In [37]:
con.sql("""
SELECT *
FROM transactions
ORDER BY sales_amount DESC
LIMIT 10"""
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod044      │ Cus020        │ Mark004     │ 2018-12-06 00:00:00 │       725 │      1510944 │ INR      │
│ Prod329      │ Cus006        │ Mark004     │ 2018-12-27 00:00:00 │       360 │      1492435 │ INR      │
│ Prod329      │ Cus006        │ Mark004     │ 2019-01-18 00:00:00 │       360 │      1477458 │ INR      │
│ Prod073      │ Cus006        │ Mark004     │ 2020-04-16 00:00:00 │       947 │      1477394 │ INR      │
│ Prod318      │ Cus038        │ Mark013     │ 2018-02-23 00:00:00 │      1798 │      1338264 │ INR      │
│ Prod316      │ Cus006        │ Mark

**Return the top 5 highest sales transactions in USD only.**

In [38]:
con.sql(
    """
    SELECT *
    FROM transactions
    WHERE currency = 'USD'
    ORDER BY sales_amount DESC
    LIMIT 5

    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-20 00:00:00 │        59 │          500 │ USD      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-22 00:00:00 │        36 │          250 │ USD      │
└──────────────┴───────────────┴─────────────┴─────────────────────┴───────────┴──────────────┴──────────┘

Find all transactions where:

sales_amount is between 5,000 and 10,000

currency is INR

In [39]:
con.sql(
    """
    SELECT *
    FROM transactions
    WHERE sales_amount BETWEEN 5000 AND 10000
      AND currency = 'INR'
    LIMIT 5

    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod002      │ Cus004        │ Mark003     │ 2018-06-18 00:00:00 │         6 │         7176 │ INR      │
│ Prod004      │ Cus005        │ Mark004     │ 2017-11-29 00:00:00 │        17 │         9426 │ INR      │
│ Prod007      │ Cus007        │ Mark004     │ 2018-10-19 00:00:00 │        11 │         6218 │ INR      │
│ Prod009      │ Cus003        │ Mark003     │ 2019-06-18 00:00:00 │         7 │         8889 │ INR      │
│ Prod009      │ Cus010        │ Mark003     │ 2019-06-28 00:00:00 │         7 │         9130 │ INR      │
└──────────────┴───────────────┴─────

List all distinct market codes from the transactions table.

In [40]:
con.sql(
    """
    SELECT DISTINCT market_code
    FROM transactions
    """
)

┌─────────────┐
│ market_code │
│   varchar   │
├─────────────┤
│ Mark001     │
│ Mark005     │
│ Mark011     │
│ Mark010     │
│ Mark002     │
│ Mark007     │
│ Mark013     │
│ Mark014     │
│ Mark004     │
│ Mark003     │
│ Mark006     │
│ Mark008     │
│ Mark015     │
│ Mark009     │
│ Mark012     │
├─────────────┤
│   15 rows   │
└─────────────┘

Question 4

Find 10 transactions where:

currency is USD
OR

sales_qty > 800

In [41]:
con.sql(
    """
    SELECT *
    FROM transactions
    WHERE currency = 'USD'
      OR sales_qty > 800
    LIMIT 10
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-20 00:00:00 │        59 │          500 │ USD      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-22 00:00:00 │        36 │          250 │ USD      │
│ Prod070      │ Cus006        │ Mark004     │ 2020-04-16 00:00:00 │       933 │       760255 │ INR      │
│ Prod073      │ Cus006        │ Mark004     │ 2020-04-16 00:00:00 │       947 │      1477394 │ INR      │
│ Prod083      │ Cus006        │ Mark004     │ 2020-02-28 00:00:00 │       933 │       562731 │ INR      │
│ Prod085      │ Cus026        │ Mark

Return the 10 smallest sales transactions (by amount).

In [42]:
con.sql(
    """
    SELECT *
    FROM transactions
    ORDER BY sales_amount ASC
    LIMIT 10
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod001      │ Cus002        │ Mark002     │ 2018-05-08 00:00:00 │         3 │           -1 │ INR      │
│ Prod001      │ Cus002        │ Mark002     │ 2018-05-08 00:00:00 │         3 │           -1 │ INR      │
│ Prod010      │ Cus003        │ Mark003     │ 2019-04-30 00:00:00 │         1 │            0 │ INR      │
│ Prod011      │ Cus018        │ Mark002     │ 2018-12-28 00:00:00 │         1 │            0 │ INR      │
│ Prod010      │ Cus015        │ Mark006     │ 2018-05-26 00:00:00 │         1 │            0 │ INR      │
│ Prod010      │ Cus003        │ Mark

Parenthesis Logic (Common Interview Trap)

Return transactions where:

currency = 'USD'
AND

(sales_amount > 10000 OR sales_qty > 1000)

This tests logical precedence.

In [43]:
con.sql("""
SELECT *
FROM transactions
WHERE currency = 'USD'
  AND (
        sales_amount > 10000
        OR sales_qty > 1000
      )
""")

┌──────────────┬───────────────┬─────────────┬──────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │  order_date  │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │ timestamp_ns │   int64   │    int64     │ varchar  │
├──────────────┴───────────────┴─────────────┴──────────────┴───────────┴──────────────┴──────────┤
│                                             0 rows                                              │
└─────────────────────────────────────────────────────────────────────────────────────────────────┘

Find transactions where:

currency is NOT NULL

AND sales_amount is greater than 0

In [44]:
con.sql("""
SELECT *
FROM transactions
WHERE currency IS NOT NULL
  AND sales_amount > 0
LIMIT 10
""")

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod001      │ Cus001        │ Mark001     │ 2017-10-10 00:00:00 │       100 │        41241 │ INR      │
│ Prod002      │ Cus003        │ Mark003     │ 2018-04-06 00:00:00 │         1 │          875 │ INR      │
│ Prod002      │ Cus003        │ Mark003     │ 2018-04-11 00:00:00 │         1 │          583 │ INR      │
│ Prod002      │ Cus004        │ Mark003     │ 2018-06-18 00:00:00 │         6 │         7176 │ INR      │
│ Prod003      │ Cus005        │ Mark004     │ 2017-11-20 00:00:00 │        59 │          500 │ USD      │
│ Prod003      │ Cus005        │ Mark

Question 3 — Multi-column Sorting

Return top 10 transactions sorted by:

sales_amount DESC

sales_qty DESC

This tests sorting priority.

In [45]:
con.sql("""
SELECT *
FROM transactions
ORDER BY sales_amount DESC, sales_qty DESC

""")

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod044      │ Cus020        │ Mark004     │ 2018-12-06 00:00:00 │       725 │      1510944 │ INR      │
│ Prod329      │ Cus006        │ Mark004     │ 2018-12-27 00:00:00 │       360 │      1492435 │ INR      │
│ Prod329      │ Cus006        │ Mark004     │ 2019-01-18 00:00:00 │       360 │      1477458 │ INR      │
│ Prod073      │ Cus006        │ Mark004     │ 2020-04-16 00:00:00 │       947 │      1477394 │ INR      │
│ Prod318      │ Cus038        │ Mark013     │ 2018-02-23 00:00:00 │      1798 │      1338264 │ INR      │
│ Prod316      │ Cus006        │ Mark

Detect Potential Data Issues

Find transactions where:

sales_amount <= 0
OR

sales_qty <= 0

This tests anomaly detection thinking.

In [46]:
con.sql("""
SELECT *
FROM transactions
WHERE sales_amount <= 0
  OR sales_qty <=0

""")

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod001      │ Cus002        │ Mark002     │ 2018-05-08 00:00:00 │         3 │           -1 │ INR      │
│ Prod010      │ Cus015        │ Mark006     │ 2018-05-26 00:00:00 │         1 │            0 │ INR      │
│ Prod010      │ Cus003        │ Mark003     │ 2019-04-30 00:00:00 │         1 │            0 │ INR      │
│ Prod011      │ Cus018        │ Mark002     │ 2018-12-28 00:00:00 │         1 │            0 │ INR      │
│ Prod001      │ Cus002        │ Mark002     │ 2018-05-08 00:00:00 │         3 │           -1 │ INR      │
│ Prod010      │ Cus015        │ Mark

Controlled Data Sampling

Return 20 transactions:

Only from INR

Sorted by order_date descending

But only select: product_code, sales_amount, order_date

Tests projection + filtering + ordering.

In [47]:
con.sql(
    """
    SELECT product_code, sales_amount, order_date
    FROM transactions
    WHERE currency = 'INR'
    ORDER BY order_date DESC
    LIMIT 20
    """
)

┌──────────────┬──────────────┬─────────────────────┐
│ product_code │ sales_amount │     order_date      │
│   varchar    │    int64     │    timestamp_ns     │
├──────────────┼──────────────┼─────────────────────┤
│ Prod318      │          375 │ 2020-06-26 00:00:00 │
│ Prod338      │          815 │ 2020-06-26 00:00:00 │
│ Prod061      │         1074 │ 2020-06-26 00:00:00 │
│ Prod065      │          454 │ 2020-06-26 00:00:00 │
│ Prod121      │           65 │ 2020-06-26 00:00:00 │
│ Prod131      │          296 │ 2020-06-26 00:00:00 │
│ Prod266      │          338 │ 2020-06-26 00:00:00 │
│ Prod055      │        13384 │ 2020-06-25 00:00:00 │
│ Prod058      │         1722 │ 2020-06-25 00:00:00 │
│ Prod065      │          282 │ 2020-06-25 00:00:00 │
│ Prod065      │        29810 │ 2020-06-25 00:00:00 │
│ Prod228      │         4037 │ 2020-06-25 00:00:00 │
│ Prod054      │         1731 │ 2020-06-24 00:00:00 │
│ Prod054      │          120 │ 2020-06-24 00:00:00 │
│ Prod059      │         436

**MODULE 2**
Total Revenue (Single Aggregation)

Adds all sales_amount

Returns one row

Changes granularity from 150k rows → 1 row

In [48]:
con.sql(
    """
    SELECT
      SUM(sales_amount) as total_revenue
    FROM transactions
    """
)

┌───────────────┐
│ total_revenue │
│    int128     │
├───────────────┤
│     986565766 │
└───────────────┘

**MODULE 2 — GROUP BY (First Level)**

1 row for entire dataset

1 row per currency

1 row per currency Per currency

Group 1 → INR rows → SUM

Group 2 → USD rows → SUM

In [49]:
con.sql(
    """
    SELECT
      currency,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY currency
    """
)

┌──────────┬───────────────┐
│ currency │ total_revenue │
│ varchar  │    int128     │
├──────────┼───────────────┤
│ INR      │     986565016 │
│ USD      │           750 │
└──────────┴───────────────┘

**Multi-Metric Aggregation**

Now we increase complexity slightly.

Instead of just total revenue, let’s calculate:

Total revenue

Total quantity

Number of transactions

All grouped by currency.

In [50]:
con.sql(
    """
    SELECT
      currency,
      SUM(sales_amount) AS total_revenue,
      SUM(sales_qty) AS total_quantity,
      COUNT(*) AS transaction_count
    FROM transactions
    GROUP BY currency;
    """
)

┌──────────┬───────────────┬────────────────┬───────────────────┐
│ currency │ total_revenue │ total_quantity │ transaction_count │
│ varchar  │    int128     │     int128     │       int64       │
├──────────┼───────────────┼────────────────┼───────────────────┤
│ INR      │     986565016 │        2444320 │            150281 │
│ USD      │           750 │             95 │                 2 │
└──────────┴───────────────┴────────────────┴───────────────────┘

MODULE 2 — Multi-Column GROUP BY
🔹 Step 3 — Revenue by Market + Currency

For each market AND each currency combination,
calculate revenue.

In [51]:
con.sql(
    """
    SELECT
      market_code,
      currency,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code, currency
    ORDER BY total_revenue DESC
    """
)

┌─────────────┬──────────┬───────────────┐
│ market_code │ currency │ total_revenue │
│   varchar   │ varchar  │    int128     │
├─────────────┼──────────┼───────────────┤
│ Mark004     │ INR      │     520720384 │
│ Mark002     │ INR      │     150180636 │
│ Mark003     │ INR      │     132526737 │
│ Mark011     │ INR      │      55026321 │
│ Mark007     │ INR      │      42128765 │
│ Mark010     │ INR      │      18813466 │
│ Mark001     │ INR      │      18227503 │
│ Mark013     │ INR      │      16525290 │
│ Mark005     │ INR      │      13583923 │
│ Mark014     │ INR      │       7436823 │
│ Mark009     │ INR      │       4428393 │
│ Mark008     │ INR      │       3094007 │
│ Mark012     │ INR      │       2605796 │
│ Mark015     │ INR      │        893857 │
│ Mark006     │ INR      │        373115 │
│ Mark004     │ USD      │           750 │
├─────────────┴──────────┴───────────────┤
│ 16 rows                      3 columns │
└────────────────────────────────────────┘

**MODULE 2 — HAVING (Filtering After Aggregation)**

🔹 Step 4 — Markets With Revenue Above 50 Million

In [52]:
con.sql(
    """
    SELECT
      market_code,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
    HAVING SUM(sales_amount) > 50000000
    ORDER BY total_revenue DESC

    """
)

┌─────────────┬───────────────┐
│ market_code │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Mark004     │     520721134 │
│ Mark002     │     150180636 │
│ Mark003     │     132526737 │
│ Mark011     │      55026321 │
└─────────────┴───────────────┘

**MODULE 2 — Conditional Aggregation**
🔹 Step 5 — Revenue by Currency in Separate Columns

In [53]:
con.sql(
    """
    SELECT
      SUM(CASE WHEN currency = 'INR' THEN sales_amount ELSE 0 END) AS inr_revenue,
      SUM(CASE WHEN currency = 'USD' THEN sales_amount ELSE 0 END) AS usd_revenue
    FROM transactions
    """
)

┌─────────────┬─────────────┐
│ inr_revenue │ usd_revenue │
│   int128    │   int128    │
├─────────────┼─────────────┤
│   986565016 │         750 │
└─────────────┴─────────────┘

**MODULE 2 — Conditional Aggregation**

🔹 Step 5 — Revenue by Currency in Separate Columns

What This Does

For each market:

Separates revenue by currency

Calculates total revenue

Orders by highest revenue market

This is dashboard-level SQL.

In [54]:
con.sql(
    """
    SELECT
    market_code,
      SUM(CASE WHEN currency = 'INR' THEN sales_amount ELSE 0 END) AS inr_revenue,
      SUM(CASE WHEN currency = 'USD' THEN sales_amount ELSE 0 END) AS usd_revenue,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
    ORDER BY total_revenue DESC;
    """
)

┌─────────────┬─────────────┬─────────────┬───────────────┐
│ market_code │ inr_revenue │ usd_revenue │ total_revenue │
│   varchar   │   int128    │   int128    │    int128     │
├─────────────┼─────────────┼─────────────┼───────────────┤
│ Mark004     │   520720384 │         750 │     520721134 │
│ Mark002     │   150180636 │           0 │     150180636 │
│ Mark003     │   132526737 │           0 │     132526737 │
│ Mark011     │    55026321 │           0 │      55026321 │
│ Mark007     │    42128765 │           0 │      42128765 │
│ Mark010     │    18813466 │           0 │      18813466 │
│ Mark001     │    18227503 │           0 │      18227503 │
│ Mark013     │    16525290 │           0 │      16525290 │
│ Mark005     │    13583923 │           0 │      13583923 │
│ Mark014     │     7436823 │           0 │       7436823 │
│ Mark009     │     4428393 │           0 │       4428393 │
│ Mark008     │     3094007 │           0 │       3094007 │
│ Mark012     │     2605796 │           

**MODULE 2 — Time-Based Aggregation**

We now bring in the dim_date table.

Until now, you only used transactions.

Now we combine:

**What This Does**

Joins fact table to date dimension

Groups revenue by year

Returns revenue per year

In [55]:
con.sql(
    """
    SELECT
      d.year,
      SUM(t.sales_amount) AS total_revenue
    FROM transactions t
    JOIN dim_date d
      ON t.order_date = d.date
    GROUP BY d.year
    ORDER BY d.year
    """
)

┌───────┬───────────────┐
│ year  │ total_revenue │
│ int64 │    int128     │
├───────┼───────────────┤
│  2017 │      93569152 │
│  2018 │     414308941 │
│  2019 │     336452114 │
│  2020 │     142235559 │
└───────┴───────────────┘

**MODULE 2 — Multi-Level Time Aggregation**

**🔹 Step 8 — Revenue by Year and Month**

In [56]:
con.sql(
    """
    SELECT
      d.year,
      d.month_name,
      SUM(t.sales_amount) AS total_revenue
    FROM transactions t
    JOIN dim_date d
      ON t.order_date = d.date
    GROUP BY d.year, d.month_name
    ORDER BY d.year, d.month_name
    """
)

┌───────┬────────────┬───────────────┐
│ year  │ month_name │ total_revenue │
│ int64 │  varchar   │    int128     │
├───────┼────────────┼───────────────┤
│  2017 │ December   │      31833907 │
│  2017 │ November   │      35385889 │
│  2017 │ October    │      26349356 │
│  2018 │ April      │      35919198 │
│  2018 │ August     │      39459625 │
│  2018 │ December   │      30405982 │
│  2018 │ February   │      35258929 │
│  2018 │ January    │      42521659 │
│  2018 │ July       │      36228538 │
│  2018 │ June       │      34925537 │
│    ·  │  ·         │          ·    │
│    ·  │  ·         │          ·    │
│    ·  │  ·         │          ·    │
│  2019 │ May        │      28050701 │
│  2019 │ November   │      26611713 │
│  2019 │ October    │      26724865 │
│  2019 │ September  │      25061571 │
│  2020 │ April      │      25264333 │
│  2020 │ February   │      26925734 │
│  2020 │ January    │      25659711 │
│  2020 │ June       │      14711976 │
│  2020 │ March      │   

**Step 9 — Fix Month Ordering Properly**

We need a numeric month column.

In [57]:
con.sql(
    """
    SELECT *
FROM dim_date
LIMIT 5;
    """
)

┌─────────────────────┬─────────────────────┬───────┬────────────┬─────────────────────┐
│        date         │       cy-date       │ year  │ month_name │     date_yy_mmm     │
│    timestamp_ns     │    timestamp_ns     │ int64 │  varchar   │    timestamp_ns     │
├─────────────────────┼─────────────────────┼───────┼────────────┼─────────────────────┤
│ 2017-06-01 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-02 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-03 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-04 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
│ 2017-06-05 00:00:00 │ 2017-06-01 00:00:00 │  2017 │ June       │ 2022-06-17 00:00:00 │
└─────────────────────┴─────────────────────┴───────┴────────────┴─────────────────────┘

**Create Month Number From Date**

Since order_date exists and dim_date.date exists, we can extract month number directly.

In BigQuery-style SQL (and DuckDB supports this):

In [58]:
con.sql(
    """
    SELECT
      d.year,
      EXTRACT(MONTH FROM d.date) AS month_number,
      d.month_name,
      SUM(sales_amount) AS total_revenue
    FROM transactions t
    JOIN dim_date d
      ON t.order_date = d.date
    GROUP BY d.year, month_number, d.month_name
    ORDER BY d.year, month_number
    """
)

┌───────┬──────────────┬────────────┬───────────────┐
│ year  │ month_number │ month_name │ total_revenue │
│ int64 │    int64     │  varchar   │    int128     │
├───────┼──────────────┼────────────┼───────────────┤
│  2017 │           10 │ October    │      26349356 │
│  2017 │           11 │ November   │      35385889 │
│  2017 │           12 │ December   │      31833907 │
│  2018 │            1 │ January    │      42521659 │
│  2018 │            2 │ February   │      35258929 │
│  2018 │            3 │ March      │      38169872 │
│  2018 │            4 │ April      │      35919198 │
│  2018 │            5 │ May        │      32273882 │
│  2018 │            6 │ June       │      34925537 │
│  2018 │            7 │ July       │      36228538 │
│    ·  │            · │  ·         │          ·    │
│    ·  │            · │  ·         │          ·    │
│    ·  │            · │  ·         │          ·    │
│  2019 │            9 │ September  │      25061571 │
│  2019 │           10 │ Oct

**Advanced grouping patterns**
What is the revenue per market?
STEP 1: Revenue per Market

In [59]:
con.sql(
    """
    SELECT
      market_code,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
    ORDER BY total_revenue DESC
    """
)

┌─────────────┬───────────────┐
│ market_code │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Mark004     │     520721134 │
│ Mark002     │     150180636 │
│ Mark003     │     132526737 │
│ Mark011     │      55026321 │
│ Mark007     │      42128765 │
│ Mark010     │      18813466 │
│ Mark001     │      18227503 │
│ Mark013     │      16525290 │
│ Mark005     │      13583923 │
│ Mark014     │       7436823 │
│ Mark009     │       4428393 │
│ Mark008     │       3094007 │
│ Mark012     │       2605796 │
│ Mark015     │        893857 │
│ Mark006     │        373115 │
├─────────────┴───────────────┤
│ 15 rows           2 columns │
└─────────────────────────────┘

Step 2
**Average of those 15 total_revenue values.**

In [60]:
con.sql(
    """
    SELECT
      AVG(total_revenue) AS avg_market_revenue
    FROM
    (
      SELECT
        market_code,
        SUM(sales_amount) AS total_revenue
      FROM transactions
      GROUP BY market_code
      ORDER BY total_revenue DESC
    ) AS market_revenue
    """
)

┌────────────────────┐
│ avg_market_revenue │
│       double       │
├────────────────────┤
│  65771051.06666667 │
└────────────────────┘

**Now Final Step — Filter Markets Above Average**

In [61]:
con.sql(
    """
    SELECT *
    FROM
    (
      SELECT
        market_code,
        SUM(sales_amount) AS total_revenue
      FROM transactions
      GROUP BY market_code
    ) AS market_revenue
    WHERE total_revenue >
    (
      SELECT
        AVG(total_revenue) AS avg_market_revenue
      FROM
      (
        SELECT
          market_code,
          SUM(sales_amount) AS total_revenue
        FROM transactions
        GROUP BY market_code
        ORDER BY total_revenue DESC
      ) AS market_revenue
    )
    """
)

┌─────────────┬───────────────┐
│ market_code │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Mark002     │     150180636 │
│ Mark003     │     132526737 │
│ Mark004     │     520721134 │
└─────────────┴───────────────┘

**Final Step**
**Clean Version Using CTE (Readable SQL)**

In [62]:
con.sql(
    """
  WITH market_revenue AS
  (
    SELECT
      market_code,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
  )

  SELECT *
  FROM market_revenue
  WHERE total_revenue >
  (
    SELECT AVG(total_revenue)
    FROM market_revenue
  )
  ORDER BY total_revenue DESC
    """
)

┌─────────────┬───────────────┐
│ market_code │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Mark004     │     520721134 │
│ Mark002     │     150180636 │
│ Mark003     │     132526737 │
└─────────────┴───────────────┘

Practice
Question 1 — Below Average Markets

Business asks:

Which markets are performing below average revenue?

You already know above average logic.

Now write:

Market_code

total_revenue

Only markets below average

Order ascending (lowest first)

In [63]:
con.sql(
    """
  WITH market_revenue AS
  (
    SELECT
      market_code,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
  )

  SELECT *
  FROM market_revenue
  WHERE total_revenue <
  (
    SELECT AVG(total_revenue)
    FROM market_revenue
  )
  ORDER BY total_revenue ASC
    """
)

┌─────────────┬───────────────┐
│ market_code │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Mark006     │        373115 │
│ Mark015     │        893857 │
│ Mark012     │       2605796 │
│ Mark008     │       3094007 │
│ Mark009     │       4428393 │
│ Mark014     │       7436823 │
│ Mark005     │      13583923 │
│ Mark013     │      16525290 │
│ Mark001     │      18227503 │
│ Mark010     │      18813466 │
│ Mark007     │      42128765 │
│ Mark011     │      55026321 │
├─────────────┴───────────────┤
│ 12 rows           2 columns │
└─────────────────────────────┘

🔹 Question 2 — Markets Above 2x Average

Harder.

Which markets have revenue more than twice the average market revenue?

This tests:

Subquery reuse

Arithmetic inside WHERE

In [64]:
con.sql(
    """
  WITH market_revenue AS
  (
    SELECT
      market_code,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
  )

  SELECT *
  FROM market_revenue
  WHERE total_revenue > 2 *
  (
    SELECT AVG(total_revenue)
    FROM market_revenue
  )
  ORDER BY total_revenue ASC
    """
)

┌─────────────┬───────────────┐
│ market_code │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Mark003     │     132526737 │
│ Mark002     │     150180636 │
│ Mark004     │     520721134 │
└─────────────┴───────────────┘

Question 3 — Highest Market Revenue vs Average

Return:

Highest market revenue

Average market revenue

In the same result.

Two columns.

No hardcoding.

In [65]:
con.sql(
    """
    WITH market_revenue AS
  (
    SELECT
      market_code,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
  )
  SELECT
    MAX(total_revenue) AS highest_market_revenue,
    AVG(total_revenue) AS average_market_revenue
  FROM market_revenue
    """
)

┌────────────────────────┬────────────────────────┐
│ highest_market_revenue │ average_market_revenue │
│         int128         │         double         │
├────────────────────────┼────────────────────────┤
│              520721134 │      65771051.06666667 │
└────────────────────────┴────────────────────────┘

🔹 Question 4 — Count How Many Markets Are Above Average

Instead of listing them:

Return only:number_of_markets_above_average

In [66]:
con.sql(
    """
  WITH market_revenue AS
  (
    SELECT
      market_code,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
  )

  SELECT COUNT(*) AS number_of_markets_above_average
  FROM market_revenue
  WHERE total_revenue >
  (
    SELECT AVG(total_revenue)
    FROM market_revenue
  )

    """
)

┌─────────────────────────────────┐
│ number_of_markets_above_average │
│              int64              │
├─────────────────────────────────┤
│                               3 │
└─────────────────────────────────┘

Percentage Above Average

Return:

market_code
total_revenue
percentage_above_average

In [67]:
con.sql(
    """
  WITH market_revenue AS
  (
    SELECT
      market_code,
      SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
  )
  SELECT
    market_code,
    total_revenue,
    ROUND(
        ((total_revenue - AVG(total_revenue) OVER())
        / AVG(total_revenue) OVER()) * 100,
        2
    ) AS percentage_above_average
  FROM market_revenue
  ORDER BY total_revenue ASC
    """
  )

┌─────────────┬───────────────┬──────────────────────────┐
│ market_code │ total_revenue │ percentage_above_average │
│   varchar   │    int128     │          double          │
├─────────────┼───────────────┼──────────────────────────┤
│ Mark006     │        373115 │                   -99.43 │
│ Mark015     │        893857 │                   -98.64 │
│ Mark012     │       2605796 │                   -96.04 │
│ Mark008     │       3094007 │                    -95.3 │
│ Mark009     │       4428393 │                   -93.27 │
│ Mark014     │       7436823 │                   -88.69 │
│ Mark005     │      13583923 │                   -79.35 │
│ Mark013     │      16525290 │                   -74.87 │
│ Mark001     │      18227503 │                   -72.29 │
│ Mark010     │      18813466 │                    -71.4 │
│ Mark007     │      42128765 │                   -35.95 │
│ Mark011     │      55026321 │                   -16.34 │
│ Mark003     │     132526737 │                    101.5

**Question 5**
Percentage Above Average

Return:

market_code, total_revenue, percentage_above_average

Without OVER() Using Join

In [68]:
con.sql(
"""
WITH market_revenue AS
(
    SELECT
        market_code,
        SUM(sales_amount) AS total_revenue
    FROM transactions
    GROUP BY market_code
),
avg_rev AS
(
    SELECT AVG(total_revenue) AS avg_revenue
    FROM market_revenue
)

SELECT
    m.market_code,
    m.total_revenue,
    ROUND(((m.total_revenue - a.avg_revenue) / a.avg_revenue) * 100, 2)
        AS percentage_above_average
FROM market_revenue m
CROSS JOIN avg_rev a
WHERE m.total_revenue > a.avg_revenue
ORDER BY percentage_above_average ASC
"""
)

┌─────────────┬───────────────┬──────────────────────────┐
│ market_code │ total_revenue │ percentage_above_average │
│   varchar   │    int128     │          double          │
├─────────────┼───────────────┼──────────────────────────┤
│ Mark003     │     132526737 │                    101.5 │
│ Mark002     │     150180636 │                   128.34 │
│ Mark004     │     520721134 │                   691.72 │
└─────────────┴───────────────┴──────────────────────────┘

**Correlated Subquery = Depends on Outer Row**

It runs once per outer row.

🟢 Business Question

**List markets whose revenue is above the average revenue of all OTHER markets.**

This is harder.

Not above global average.

Above the average of the rest.


Concept

For each market:

**Compute its revenue.**

**Compute average revenue of all other markets (excluding itself).**

**Compare**.

That means:

The inner query must reference the outer row.

That is correlation.

**Step 1 — Build Revenue Per Market (Base Layer)**

We reuse:

In [69]:
con.sql(
    """
    WITH market_revenue AS (
  SELECT
    market_code,
    SUM(sales_amount) AS total_revenue
  FROM transactions
  GROUP BY market_code
)

SELECT *
FROM market_revenue
WHERE total_revenue >
(
  SELECT AVG(total_revenue)
  FROM market_revenue
  WHERE total_revenue < (
      SELECT MAX(total_revenue)
      FROM market_revenue
  )
)
    """
)

┌─────────────┬───────────────┐
│ market_code │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Mark004     │     520721134 │
│ Mark003     │     132526737 │
│ Mark011     │      55026321 │
│ Mark002     │     150180636 │
│ Mark007     │      42128765 │
└─────────────┴───────────────┘

**EXISTS / NOT EXISTS**

**This is heavily used in:**

Data validation

Integrity checks

Filtering based on related tables

Performance-safe filtering

And it behaves differently from JOIN.

**Business Question 1**

List customers who have made at least one transaction.

Think first.

We have:

customers table

transactions table

We want:

Customers for whom a matching transaction exists.

This is a perfect EXISTS case.

In [70]:
con.sql(
    """
    SELECT *
    FROM customers c
    WHERE EXISTS(
      SELECT 1
      FROM transactions t
      WHERE t.customer_code = c.customer_code
    )
    """
)

┌───────────────┬─────────────────────────┬────────────────┐
│ customer_code │      customer_name      │ customer_type  │
│    varchar    │         varchar         │    varchar     │
├───────────────┼─────────────────────────┼────────────────┤
│ Cus007        │ Info Stores             │ Brick & Mortar │
│ Cus018        │ Electricalslance Stores │ Brick & Mortar │
│ Cus003        │ Excel Stores            │ Brick & Mortar │
│ Cus024        │ Power                   │ E-Commerce     │
│ Cus031        │ Zone                    │ E-Commerce     │
│ Cus013        │ Unity Stores            │ Brick & Mortar │
│ Cus029        │ Electricalsocity        │ E-Commerce     │
│ Cus016        │ Logic Stores            │ Brick & Mortar │
│ Cus023        │ Sound                   │ E-Commerce     │
│ Cus025        │ Path                    │ E-Commerce     │
│   ·           │  ·                      │     ·          │
│   ·           │  ·                      │     ·          │
│   ·           │  ·    

In [71]:
con.sql(
    """
    SELECT DISTINCT c.*
    FROM customers c
    JOIN transactions t
    ON t.customer_code = c.customer_code
    """
)

┌───────────────┬─────────────────────────┬────────────────┐
│ customer_code │      customer_name      │ customer_type  │
│    varchar    │         varchar         │    varchar     │
├───────────────┼─────────────────────────┼────────────────┤
│ Cus008        │ Acclaimed Stores        │ Brick & Mortar │
│ Cus020        │ Nixon                   │ E-Commerce     │
│ Cus031        │ Zone                    │ E-Commerce     │
│ Cus017        │ Epic Stores             │ Brick & Mortar │
│ Cus002        │ Nomad Stores            │ Brick & Mortar │
│ Cus018        │ Electricalslance Stores │ Brick & Mortar │
│ Cus024        │ Power                   │ E-Commerce     │
│ Cus033        │ All-Out                 │ E-Commerce     │
│ Cus023        │ Sound                   │ E-Commerce     │
│ Cus030        │ Synthetic               │ E-Commerce     │
│   ·           │   ·                     │     ·          │
│   ·           │   ·                     │     ·          │
│   ·           │   ·   

**NOT EXISTS (Very Important for Data Quality)**

Business question:

Which customers have NEVER made a transaction?

This is extremely common in analytics and data engineering.

Write the query using NOT EXISTS.

In [72]:
con.sql(
    """
    SELECT *
    FROM customers c
    WHERE NOT EXISTS(
      SELECT 1
      FROM transactions t
      WHERE c.customer_code = t.customer_code
    )
    """
)

┌───────────────┬───────────────┬───────────────┐
│ customer_code │ customer_name │ customer_type │
│    varchar    │    varchar    │    varchar    │
├───────────────┴───────────────┴───────────────┤
│                    0 rows                     │
└───────────────────────────────────────────────┘

In [73]:
con.sql(
    """
    SELECT *
    FROM transactions t
    WHERE NOT EXISTS (
    SELECT 1
    FROM customers c
    WHERE c.customer_code = t.customer_code
);
    """
)

┌──────────────┬───────────────┬─────────────┬──────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │  order_date  │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │ timestamp_ns │   int64   │    int64     │ varchar  │
├──────────────┴───────────────┴─────────────┴──────────────┴───────────┴──────────────┴──────────┤
│                                             0 rows                                              │
└─────────────────────────────────────────────────────────────────────────────────────────────────┘

**Complex EXISTS Pattern 1**
“Customers Who Bought From More Than One Market”

Business question:

Which customers have transactions in more than one market?

In [74]:
con.sql(
    """
    SELECT *
FROM customers c
WHERE EXISTS (
    SELECT 1
    FROM transactions t1
    WHERE t1.customer_code = c.customer_code
)
AND EXISTS (
    SELECT 1
    FROM transactions t2
    WHERE t2.customer_code = c.customer_code
    AND t2.market_code <> (
        SELECT t3.market_code
        FROM transactions t3
        WHERE t3.customer_code = c.customer_code
        LIMIT 1
    )
)
    """
)

┌───────────────┬─────────────────────────┬────────────────┐
│ customer_code │      customer_name      │ customer_type  │
│    varchar    │         varchar         │    varchar     │
├───────────────┼─────────────────────────┼────────────────┤
│ Cus007        │ Info Stores             │ Brick & Mortar │
│ Cus018        │ Electricalslance Stores │ Brick & Mortar │
│ Cus003        │ Excel Stores            │ Brick & Mortar │
│ Cus024        │ Power                   │ E-Commerce     │
│ Cus031        │ Zone                    │ E-Commerce     │
│ Cus013        │ Unity Stores            │ Brick & Mortar │
│ Cus029        │ Electricalsocity        │ E-Commerce     │
│ Cus016        │ Logic Stores            │ Brick & Mortar │
│ Cus023        │ Sound                   │ E-Commerce     │
│ Cus025        │ Path                    │ E-Commerce     │
│   ·           │  ·                      │     ·          │
│   ·           │  ·                      │     ·          │
│   ·           │  ·    

**Professional Cleaner Way (Group By Version)**

The clean version is:

In [75]:
con.sql("""SELECT customer_code
FROM transactions
GROUP BY customer_code
HAVING COUNT(DISTINCT market_code) > 1
""")

┌───────────────┐
│ customer_code │
│    varchar    │
├───────────────┤
│ Cus018        │
│ Cus033        │
│ Cus011        │
│ Cus034        │
│ Cus016        │
│ Cus023        │
│ Cus038        │
│ Cus010        │
│ Cus027        │
│ Cus014        │
│   ·           │
│   ·           │
│   ·           │
│ Cus025        │
│ Cus029        │
│ Cus017        │
│ Cus012        │
│ Cus003        │
│ Cus015        │
│ Cus035        │
│ Cus024        │
│ Cus021        │
│ Cus037        │
├───────────────┤
│    38 rows    │
│  (20 shown)   │
└───────────────┘

**WINDOW FUNCTIONS**

1️⃣ CONCEPT — ROW_NUMBER()
What is ROW_NUMBER()?

ROW_NUMBER() is a window function that assigns a unique sequential number to each row within a partition.

It does NOT collapse rows.

It does NOT aggregate.

It only labels rows.

**Why does it exist?**

Because in analytics and data engineering we often need:

Top-N per group

Deduplication

Latest record selection

Change tracking

Deterministic ordering

GROUP BY cannot do this.

**What problem does it solve?**

It allows us to:

Rank or label rows inside logical groups without losing row-level detail.

This is critical for:

Keeping latest customer record

Selecting best-performing product per market

Removing duplicates in staging

Implementing SCD logic

**Where is it used in Data Engineering?**

Very common in:

Staging layer deduplication

CDC merge pipelines

Snapshot creation

Identifying first/last transaction

Incremental loading logic

What is the highest revenue transaction in each market?”

**First Approach (Without Window Functions)**

In [76]:
con.sql(
    """
    SELECT t.*
    FROM transactions t
    JOIN(
      SELECT market_code, MAX(sales_amount) AS max_sales
      FROM transactions
      GROUP BY market_code
    )m
    ON t.market_code = m.market_code
    AND t.sales_amount = m.max_sales
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┤
│ Prod318      │ Cus038        │ Mark013     │ 2018-02-23 00:00:00 │      1798 │      1338264 │ INR      │
│ Prod318      │ Cus027        │ Mark012     │ 2018-04-26 00:00:00 │       187 │        83222 │ INR      │
│ Prod318      │ Cus027        │ Mark012     │ 2018-06-12 00:00:00 │       187 │        83222 │ INR      │
│ Prod318      │ Cus027        │ Mark012     │ 2018-06-19 00:00:00 │       187 │        83222 │ INR      │
│ Prod319      │ Cus001        │ Mark001     │ 2018-09-10 00:00:00 │       637 │       393866 │ INR      │
│ Prod018      │ Cus026        │ Mark

Now with Window Function
🔹Introduce **ROW_NUMBER()**

In [77]:
con.sql(
    """
    SELECT *
    FROM(
      SELECT
        t.*,
        ROW_NUMBER() Over(
          PARTITION BY market_code
          ORDER BY sales_amount DESC
        )AS rn
      FROM transactions t
    ) ranked
    WHERE rn = 1
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┬───────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │  rn   │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │ int64 │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┼───────┤
│ Prod232      │ Cus028        │ Mark003     │ 2018-01-31 00:00:00 │      8800 │       647778 │ INR      │     1 │
│ Prod318      │ Cus027        │ Mark012     │ 2018-04-26 00:00:00 │       187 │        83222 │ INR      │     1 │
│ Prod319      │ Cus001        │ Mark001     │ 2018-09-10 00:00:00 │       637 │       393866 │ INR      │     1 │
│ Prod040      │ Cus008        │ Mark005     │ 2018-01-23 00:00:00 │       240 │       410000 │ INR      │     1 │
│ Prod024      │ Cus011        │ Mark006     │ 2018-03-12 00:00:00 │       132 │

**Deduplication Using ROW_NUMBER()**

This is where window functions stop being “analytics” and start being pipeline logic.

🔹 1️⃣ CONCEPT — Deduplication
Problem

In real systems:

Source systems resend data

Batch jobs rerun

APIs retry

Late-arriving data appears

Result:

Duplicate records in staging tables.

Keep only one record per (product_code, customer_code, order_date)

Since your dataset has no last_updated column,
we simulate keeping highest sales_amount.

In [78]:
con.sql(
    """
    SELECT *
    FROM(
      SELECT t.*,
      ROW_NUMBER() OVER(
        PARTITION BY product_code, customer_code, order_date
        ORDER BY sales_amount DESC
      ) AS rn
    FROM transactions t
  )deduped
  WHERE rn = 1
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┬───────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │  rn   │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │ int64 │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┼───────┤
│ Prod300      │ Cus007        │ Mark004     │ 2020-06-08 00:00:00 │         1 │           93 │ INR      │     1 │
│ Prod300      │ Cus011        │ Mark003     │ 2017-11-29 00:00:00 │         1 │           42 │ INR      │     1 │
│ Prod300      │ Cus011        │ Mark003     │ 2018-05-16 00:00:00 │         1 │           42 │ INR      │     1 │
│ Prod300      │ Cus011        │ Mark003     │ 2018-06-25 00:00:00 │         2 │          204 │ INR      │     1 │
│ Prod300      │ Cus011        │ Mark003     │ 2018-09-28 00:00:00 │         1 │

In [79]:
con.sql(
    """
    SELECT
    product_code,
    customer_code,
    order_date,
    COUNT(*) AS cnt
FROM (
    SELECT
        t.*,
        ROW_NUMBER() OVER (
            PARTITION BY product_code, customer_code, order_date
            ORDER BY sales_amount DESC
        ) AS rn
    FROM transactions t
) x
WHERE rn = 1
GROUP BY product_code, customer_code, order_date
HAVING COUNT(*) > 1;
    """
)

┌──────────────┬───────────────┬──────────────┬───────┐
│ product_code │ customer_code │  order_date  │  cnt  │
│   varchar    │    varchar    │ timestamp_ns │ int64 │
├──────────────┴───────────────┴──────────────┴───────┤
│                       0 rows                        │
└─────────────────────────────────────────────────────┘

Write SQL to:

Get the latest transaction per customer in each market.

In [80]:
con.sql(
    """
    SELECT
      customer_code,
      market_code,
      order_date,
      sales_amount
    FROM (
      SELECT
          t.*,
          ROW_NUMBER() OVER (
              PARTITION BY customer_code, market_code
              ORDER BY order_date DESC
          ) AS rn
      FROM transactions t
    )latest_txn
WHERE rn = 1
ORDER BY customer_code, market_code
    """
)

┌───────────────┬─────────────┬─────────────────────┬──────────────┐
│ customer_code │ market_code │     order_date      │ sales_amount │
│    varchar    │   varchar   │    timestamp_ns     │    int64     │
├───────────────┼─────────────┼─────────────────────┼──────────────┤
│ Cus001        │ Mark001     │ 2020-06-22 00:00:00 │        59801 │
│ Cus001        │ Mark010     │ 2020-06-18 00:00:00 │         3458 │
│ Cus002        │ Mark002     │ 2020-06-16 00:00:00 │         6838 │
│ Cus002        │ Mark005     │ 2020-06-18 00:00:00 │         2769 │
│ Cus002        │ Mark006     │ 2018-08-16 00:00:00 │            0 │
│ Cus002        │ Mark007     │ 2020-06-16 00:00:00 │          389 │
│ Cus003        │ Mark002     │ 2020-06-01 00:00:00 │       168167 │
│ Cus003        │ Mark003     │ 2020-06-26 00:00:00 │          815 │
│ Cus003        │ Mark006     │ 2018-10-16 00:00:00 │            0 │
│ Cus003        │ Mark014     │ 2020-06-12 00:00:00 │          407 │
│   ·           │    ·        │   

In [81]:
con.sql(
    """
    SELECT
    customer_code,
    market_code,
    COUNT(*) AS cnt
FROM (
    SELECT
        t.*,
        ROW_NUMBER() OVER (
            PARTITION BY customer_code, market_code
            ORDER BY order_date DESC
        ) AS rn
    FROM transactions t
) x
WHERE rn = 1
GROUP BY customer_code, market_code
HAVING COUNT(*) > 1;
    """
)

┌───────────────┬─────────────┬───────┐
│ customer_code │ market_code │  cnt  │
│    varchar    │   varchar   │ int64 │
├───────────────┴─────────────┴───────┤
│               0 rows                │
└─────────────────────────────────────┘

New Business Question (Corrected)

For each market, find the customer who made the most recent transaction.

That means:

For every market_code
→ Look at all transactions
→ Find the row with the maximum order_date
→ Return that row (customer, date, sales)

In [82]:
con.sql(
    """
    SELECT
      market_code,
      customer_code,
      order_date,
      sales_amount
      FROM(
        SELECT t.*,
        ROW_NUMBER() OVER(
          PARTITION BY market_code
          ORDER BY order_date DESC
        )AS rn
        FROM transactions t
      )latest_per_market
    WHERE rn = 1
    ORDER BY market_code
    """
)

┌─────────────┬───────────────┬─────────────────────┬──────────────┐
│ market_code │ customer_code │     order_date      │ sales_amount │
│   varchar   │    varchar    │    timestamp_ns     │    int64     │
├─────────────┼───────────────┼─────────────────────┼──────────────┤
│ Mark001     │ Cus001        │ 2020-06-22 00:00:00 │        59801 │
│ Mark002     │ Cus004        │ 2020-06-17 00:00:00 │         4292 │
│ Mark003     │ Cus003        │ 2020-06-26 00:00:00 │          338 │
│ Mark004     │ Cus020        │ 2020-06-24 00:00:00 │         6440 │
│ Mark005     │ Cus027        │ 2020-06-23 00:00:00 │        50134 │
│ Mark006     │ Cus015        │ 2018-10-22 00:00:00 │            0 │
│ Mark007     │ Cus017        │ 2020-06-17 00:00:00 │         7690 │
│ Mark008     │ Cus026        │ 2020-05-25 00:00:00 │         8597 │
│ Mark009     │ Cus032        │ 2020-06-16 00:00:00 │        39843 │
│ Mark010     │ Cus004        │ 2020-06-19 00:00:00 │         2745 │
│ Mark011     │ Cus031        │ 20

In [83]:
con.sql(
    """
    SELECT
    market_code,
    customer_code,
    order_date,
    sales_amount
FROM (
    SELECT
        t.*,
        RANK() OVER (
            PARTITION BY market_code
            ORDER BY order_date DESC
        ) AS rnk
    FROM transactions t
) ranked
WHERE rnk = 1
ORDER BY market_code;
    """
)

┌─────────────┬───────────────┬─────────────────────┬──────────────┐
│ market_code │ customer_code │     order_date      │ sales_amount │
│   varchar   │    varchar    │    timestamp_ns     │    int64     │
├─────────────┼───────────────┼─────────────────────┼──────────────┤
│ Mark001     │ Cus001        │ 2020-06-22 00:00:00 │        59801 │
│ Mark002     │ Cus004        │ 2020-06-17 00:00:00 │         4292 │
│ Mark003     │ Cus003        │ 2020-06-26 00:00:00 │          338 │
│ Mark003     │ Cus003        │ 2020-06-26 00:00:00 │          815 │
│ Mark003     │ Cus003        │ 2020-06-26 00:00:00 │          375 │
│ Mark003     │ Cus003        │ 2020-06-26 00:00:00 │           65 │
│ Mark003     │ Cus003        │ 2020-06-26 00:00:00 │          296 │
│ Mark003     │ Cus003        │ 2020-06-26 00:00:00 │         1074 │
│ Mark003     │ Cus003        │ 2020-06-26 00:00:00 │          454 │
│ Mark004     │ Cus020        │ 2020-06-24 00:00:00 │         6440 │
│    ·        │   ·           │   

**RANK() vs DENSE_RANK() — Deep Understanding**

We will follow structure strictly.

🔹 1️⃣ CONCEPT

**What is RANK?**

RANK assigns ranking position within a partition based on ORDER BY.

If two rows tie:

They get the same rank.

The next rank number is skipped.

Example:

Scores: 100, 90, 90, 80
Ranks: 1, 2, 2, 4

Notice:
Rank 3 is skipped.

**What is DENSE_RANK?**

DENSE_RANK also assigns same rank to ties.

But it does NOT skip numbers.

Example:

Scores: 100, 90, 90, 80
Dense ranks: 1, 2, 2, 3

No gap.

Rank markets based on total revenue.

In [84]:
con.sql(
    """
    WITH market_revenue AS(
      SELECT
        market_code,
        SUM(sales_amount) AS total_revenue
      FROM transactions
      GROUP BY market_code
    )
    SELECT
      market_code,
      total_revenue,
      RANK() OVER(
        ORDER BY total_revenue DESC
      )AS revenue_rank,
      DENSE_RANK() OVER(
        ORDER BY total_revenue DESC
      )AS revenue_dense_rank
    FROM market_revenue
    ORDER BY total_revenue DESC
    """
)

┌─────────────┬───────────────┬──────────────┬────────────────────┐
│ market_code │ total_revenue │ revenue_rank │ revenue_dense_rank │
│   varchar   │    int128     │    int64     │       int64        │
├─────────────┼───────────────┼──────────────┼────────────────────┤
│ Mark004     │     520721134 │            1 │                  1 │
│ Mark002     │     150180636 │            2 │                  2 │
│ Mark003     │     132526737 │            3 │                  3 │
│ Mark011     │      55026321 │            4 │                  4 │
│ Mark007     │      42128765 │            5 │                  5 │
│ Mark010     │      18813466 │            6 │                  6 │
│ Mark001     │      18227503 │            7 │                  7 │
│ Mark013     │      16525290 │            8 │                  8 │
│ Mark005     │      13583923 │            9 │                  9 │
│ Mark014     │       7436823 │           10 │                 10 │
│ Mark009     │       4428393 │           11 │  

**NTILE()**
CONCEPT

**What is NTILE(n)?**

NTILE divides ordered rows into n approximately equal groups.

Think:

“Split ranked data into buckets.”

Example:

If 100 customers are sorted by revenue:

NTILE(4)

You create:

Quartile 1

Quartile 2

Quartile 3

Quartile 4

Each group roughly 25 rows.

Business Query (Atliq)

Segment customers into 4 revenue quartiles based on total sales.

In [85]:
con.sql(
    """
    WITH customer_revenue AS(
      SELECT
        customer_code,
        SUM(sales_amount) AS total_revenue
      FROM transactions
      GROUP BY customer_code

    )

    SELECT
      customer_code,
      total_revenue,
      NTILE(4) OVER(
        ORDER BY total_revenue DESC
      )AS revenue_quartile
    FROM customer_revenue
    ORDER BY total_revenue DESC
    LIMIT 20

    """
)

┌───────────────┬───────────────┬──────────────────┐
│ customer_code │ total_revenue │ revenue_quartile │
│    varchar    │    int128     │      int64       │
├───────────────┼───────────────┼──────────────────┤
│ Cus006        │     413905769 │                1 │
│ Cus022        │      49644189 │                1 │
│ Cus003        │      49175285 │                1 │
│ Cus005        │      45258250 │                1 │
│ Cus020        │      43916981 │                1 │
│ Cus007        │      35359233 │                1 │
│ Cus027        │      31771997 │                1 │
│ Cus001        │      28833717 │                1 │
│ Cus008        │      21198041 │                1 │
│ Cus014        │      21079123 │                1 │
│ Cus017        │      18794634 │                2 │
│ Cus002        │      17739349 │                2 │
│ Cus029        │      17489935 │                2 │
│ Cus021        │      17379851 │                2 │
│ Cus010        │      16716803 │             

**SUM() OVER() — Running & Cumulative Analytics**

This is one of the most important analytical SQL patterns used in dashboards, financial reporting, and time-series pipelines.

**What is SUM() OVER()?**

SUM() OVER() computes a running or partition-based aggregation without collapsing rows.

Sometimes business needs:

“Show each transaction AND cumulative sales up to that point.”

That means:

**We must retain each row, while also calculating a cumulative value.**

This is exactly what window functions allow.

Business Context (Atliq Dataset)

Business question:

**For each market, show the cumulative sales over time.**

Meaning:

Market M1 → running total of sales by date

Market M2 → running total of sales by date

Each market should accumulate independently.

In [86]:
con.sql(
    """
    SELECT
      market_code,
      order_date,
      sales_amount,
      SUM(sales_amount) OVER(
        PARTITION by market_code
        ORDER BY order_date
        ) AS cumulative_sales
    FROM transactions
    ORDER BY market_code, order_date
    LIMIT 30
    """
)

┌─────────────┬─────────────────────┬──────────────┬──────────────────┐
│ market_code │     order_date      │ sales_amount │ cumulative_sales │
│   varchar   │    timestamp_ns     │    int64     │      int128      │
├─────────────┼─────────────────────┼──────────────┼──────────────────┤
│ Mark001     │ 2017-10-10 00:00:00 │        22167 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │        41241 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │       143560 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │           51 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │         3556 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │        41241 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │         2685 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │        33231 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │         1190 │           700960 │
│ Mark001     │ 2017-10-10 00:00:00 │          162 │           7

**CORRECT QUERY**

In [87]:
con.sql(
    """
    SELECT
      market_code,
      order_date,
      sales_amount,
      SUM(sales_amount) OVER (
          PARTITION BY market_code
          ORDER BY order_date, sales_amount
          ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
      ) AS rolling_sales
    FROM transactions
    ORDER BY market_code, order_date;
    """
)

┌─────────────┬─────────────────────┬──────────────┬───────────────┐
│ market_code │     order_date      │ sales_amount │ rolling_sales │
│   varchar   │    timestamp_ns     │    int64     │    int128     │
├─────────────┼─────────────────────┼──────────────┼───────────────┤
│ Mark001     │ 2017-10-10 00:00:00 │           51 │            51 │
│ Mark001     │ 2017-10-10 00:00:00 │          162 │           213 │
│ Mark001     │ 2017-10-10 00:00:00 │          208 │           421 │
│ Mark001     │ 2017-10-10 00:00:00 │         1190 │          1560 │
│ Mark001     │ 2017-10-10 00:00:00 │         1431 │          2829 │
│ Mark001     │ 2017-10-10 00:00:00 │         2685 │          5306 │
│ Mark001     │ 2017-10-10 00:00:00 │         3556 │          7672 │
│ Mark001     │ 2017-10-10 00:00:00 │        22167 │         28408 │
│ Mark001     │ 2017-10-10 00:00:00 │        33231 │         58954 │
│ Mark001     │ 2017-10-10 00:00:00 │        39778 │         95176 │
│    ·        │          ·        

**CONCEPT — LAG() and LEAD()**

These functions allow you to access values from other rows relative to the current row.

They do not aggregate.
They peek into nearby rows.

**LAG() Returns a value from a previous row.**

Example idea:

today_sales
yesterday_sales

**LEAD() Returns a value from a next row.**

Example idea:

current_month_sales
next_month_sales

**SYNTAX**

LAG(column, offset, default)

OVER (
    PARTITION BY ...
    ORDER BY ...

Business Question 1

Calculate daily revenue per market and show the previous day's revenue.

This requires:

1️⃣ Aggregation

2️⃣ Join with date dimension (optional but common in warehouses)

3️⃣ LAG window

**Step 1 — Aggregate Revenue**
**Step 2 — Apply LAG**

In [88]:
con.sql(
    """
    WITH daily_sales AS(
      SELECT
        market_code,
        order_date,
        SUM(sales_amount) AS daily_revenue
      FROM transactions
      GROUP BY market_code, order_date
    )
    SELECT
      market_code,
      order_date,
      daily_revenue,
      LAG(daily_revenue) OVER(
        PARTITION BY market_code
        ORDER BY order_date
      )AS privious_date_revenue,

      daily_revenue -
      LAG(daily_revenue) OVER (
        PARTITION BY market_code
        ORDER BY order_date
    ) AS revenue_change
    FROM daily_sales
    ORDER BY market_code, order_date
    """
)

┌─────────────┬─────────────────────┬───────────────┬───────────────────────┬────────────────┐
│ market_code │     order_date      │ daily_revenue │ privious_date_revenue │ revenue_change │
│   varchar   │    timestamp_ns     │    int128     │        int128         │     int128     │
├─────────────┼─────────────────────┼───────────────┼───────────────────────┼────────────────┤
│ Mark001     │ 2017-10-10 00:00:00 │        700960 │                  NULL │           NULL │
│ Mark001     │ 2017-10-20 00:00:00 │        152677 │                700960 │        -548283 │
│ Mark001     │ 2017-10-25 00:00:00 │         85876 │                152677 │         -66801 │
│ Mark001     │ 2017-10-27 00:00:00 │        123189 │                 85876 │          37313 │
│ Mark001     │ 2017-11-03 00:00:00 │         37866 │                123189 │         -85323 │
│ Mark001     │ 2017-11-06 00:00:00 │         91651 │                 37866 │          53785 │
│ Mark001     │ 2017-11-14 00:00:00 │         5338

Business Question

**Find markets where revenue dropped compared to the previous day.**

In [89]:
con.sql(
    """
    WITH daily_sales AS (
    SELECT
      market_code,
      order_date,
      SUM(sales_amount) AS daily_revenue
    FROM transactions
    GROUP BY market_code, order_date
),

sales_with_lag AS (
    SELECT
      market_code,
      order_date,
      daily_revenue,
      LAG(daily_revenue) OVER (
          PARTITION BY market_code
          ORDER BY order_date
          ) AS previous_day_revenue
    FROM daily_sales
)

  SELECT *
  FROM sales_with_lag
  WHERE daily_revenue < previous_day_revenue
  ORDER BY market_code, order_date;
    """
)

┌─────────────┬─────────────────────┬───────────────┬──────────────────────┐
│ market_code │     order_date      │ daily_revenue │ previous_day_revenue │
│   varchar   │    timestamp_ns     │    int128     │        int128        │
├─────────────┼─────────────────────┼───────────────┼──────────────────────┤
│ Mark001     │ 2017-10-20 00:00:00 │        152677 │               700960 │
│ Mark001     │ 2017-10-25 00:00:00 │         85876 │               152677 │
│ Mark001     │ 2017-11-03 00:00:00 │         37866 │               123189 │
│ Mark001     │ 2017-11-14 00:00:00 │         53389 │                91651 │
│ Mark001     │ 2017-11-16 00:00:00 │         39591 │                53389 │
│ Mark001     │ 2017-12-11 00:00:00 │         37348 │                39591 │
│ Mark001     │ 2017-12-27 00:00:00 │         22204 │               326079 │
│ Mark001     │ 2018-01-10 00:00:00 │        113912 │               588647 │
│ Mark001     │ 2018-01-26 00:00:00 │        164111 │               368013 │

Next Level Business Problem (Harder)

**Now we introduce joins + LAG + ranking.**

Business Question

Find the top 3 markets that experienced the largest revenue drop day-over-day.

In [90]:
con.sql(
    """
    WITH daily_sales AS (
    SELECT
      market_code,
      order_date,
      SUM(sales_amount) AS daily_revenue
    FROM transactions
    GROUP BY market_code, order_date
),

sales_with_previous AS (
    SELECT
      market_code,
      order_date,
      daily_revenue,
      LAG(daily_revenue) OVER (
          PARTITION BY market_code
          ORDER BY order_date
      ) AS previous_day_revenue
    FROM daily_sales
),

revenue_change AS (
    SELECT
      market_code,
      order_date,
      daily_revenue,
      previous_day_revenue,
      previous_day_revenue - daily_revenue AS revenue_drop
    FROM sales_with_previous
),

ranked_drops AS (
    SELECT
      market_code,
      order_date,
      revenue_drop,
      RANK() OVER (
          ORDER BY revenue_drop DESC
      ) AS drop_rank
    FROM revenue_change
)

SELECT *
FROM ranked_drops
WHERE drop_rank <= 3;
    """
)

┌─────────────┬─────────────────────┬──────────────┬───────────┐
│ market_code │     order_date      │ revenue_drop │ drop_rank │
│   varchar   │    timestamp_ns     │    int128    │   int64   │
├─────────────┼─────────────────────┼──────────────┼───────────┤
│ Mark004     │ 2019-02-11 00:00:00 │      3402669 │         1 │
│ Mark004     │ 2018-12-17 00:00:00 │      3319753 │         2 │
│ Mark004     │ 2018-03-01 00:00:00 │      3273987 │         3 │
└─────────────┴─────────────────────┴──────────────┴───────────┘

Write a query for this:

Find the top 5 markets with the largest revenue increase compared to the previous day.

In [91]:
con.sql(
    """
    WITH daily_sales AS (
      SELECT
        market_code,
        order_date,
        SUM(sales_amount) AS daily_revenue
      FROM transactions
      GROUP BY market_code, order_date
    ),

    sales_with_previous AS(
      SELECT
        market_code,
        order_date,
        daily_revenue,

        LAG(daily_revenue) OVER(
          PARTITION BY market_code
          ORDER BY order_date
        ) AS previous_day_revenue
      FROM daily_sales
    ),

    revenue_change AS (
      SELECT
        market_code,
        order_date,
        daily_revenue,
        previous_day_revenue,

        daily_revenue - previous_day_revenue AS revenue_rise
      FROM sales_with_previous
      WHERE previous_day_revenue IS NOT NULL
    ),

    ranked_rise AS (
      SELECT
        market_code,
        order_date,
        revenue_rise,

        RANK() OVER(
          ORDER BY revenue_rise DESC
        )AS rise_rank
      FROM revenue_change
    )

    SELECT *
    FROM ranked_rise
    WHERE rise_rank <= 5
    """
)

┌─────────────┬─────────────────────┬──────────────┬───────────┐
│ market_code │     order_date      │ revenue_rise │ rise_rank │
│   varchar   │    timestamp_ns     │    int128    │   int64   │
├─────────────┼─────────────────────┼──────────────┼───────────┤
│ Mark004     │ 2018-01-19 00:00:00 │      3647013 │         1 │
│ Mark004     │ 2019-02-08 00:00:00 │      3516749 │         2 │
│ Mark004     │ 2018-12-14 00:00:00 │      3436734 │         3 │
│ Mark004     │ 2018-01-30 00:00:00 │      3423049 │         4 │
│ Mark004     │ 2017-10-13 00:00:00 │      2944002 │         5 │
└─────────────┴─────────────────────┴──────────────┴───────────┘

Next Level Problem (More Realistic)

Now we introduce joins + windows + ranking.

Business Question

Find the top 3 customers in each market based on total revenue.

In [92]:
con.sql(
    """
    WITH customer_market_sales AS (
      SELECT
        market_code,
        customer_code,
        SUM(sales_amount) AS total_revenue
      FROM transactions
      GROUP BY market_code, customer_code
),

customer_ranked AS (
      SELECT
        market_code,
        customer_code,
        total_revenue,

        RANK() OVER(
            PARTITION BY market_code
            ORDER BY total_revenue DESC
        ) AS customer_rank

      FROM customer_market_sales
)

SELECT *
FROM customer_ranked
WHERE customer_rank <= 3
ORDER BY market_code, customer_rank
LIMIT 20
    """
)

┌─────────────┬───────────────┬───────────────┬───────────────┐
│ market_code │ customer_code │ total_revenue │ customer_rank │
│   varchar   │    varchar    │    int128     │     int64     │
├─────────────┼───────────────┼───────────────┼───────────────┤
│ Mark001     │ Cus001        │      18227503 │             1 │
│ Mark002     │ Cus022        │      48095673 │             1 │
│ Mark002     │ Cus029        │      17050353 │             2 │
│ Mark002     │ Cus008        │      13336103 │             3 │
│ Mark003     │ Cus003        │      31481318 │             1 │
│ Mark003     │ Cus027        │      29097266 │             2 │
│ Mark003     │ Cus010        │      16425710 │             3 │
│ Mark004     │ Cus006        │     411108413 │             1 │
│ Mark004     │ Cus005        │      41081401 │             2 │
│ Mark004     │ Cus020        │      36700084 │             3 │
│ Mark005     │ Cus008        │       7861938 │             1 │
│ Mark005     │ Cus033        │       38

Now Let's Make It More Realistic (Add Join)

If we want customer names instead of codes:

Find top 3 customers in each market based on revenue, showing customer and market names.

In [93]:
con.sql(
    """
  WITH customer_sales AS (
  SELECT
    m.markets_name,
    c.customer_name,
    SUM(t.sales_amount) AS total_revenue
  FROM transactions t

  JOIN customers c
  ON t.customer_code = c.customer_code

  JOIN markets m
  ON t.market_code = m.market_code

  GROUP BY
    m.markets_name,
    c.customer_name
),

customer_ranked AS (
  SELECT
    markets_name,
    customer_name,
    total_revenue,

    RANK() OVER(
        PARTITION BY markets_name
        ORDER BY total_revenue DESC
    ) AS rank

FROM customer_sales
)

SELECT *
FROM customer_ranked
WHERE rank <= 3
ORDER BY markets_name, rank
LIMIT 20
    """
)

┌──────────────┬───────────────────────┬───────────────┬───────┐
│ markets_name │     customer_name     │ total_revenue │ rank  │
│   varchar    │        varchar        │    int128     │ int64 │
├──────────────┼───────────────────────┼───────────────┼───────┤
│ Ahmedabad    │ Excel Stores          │      31481318 │     1 │
│ Ahmedabad    │ Control               │      29097266 │     2 │
│ Ahmedabad    │ Atlas Stores          │      16425710 │     3 │
│ Bengaluru    │ Flawless Stores       │        290750 │     1 │
│ Bengaluru    │ Epic Stores           │         49523 │     2 │
│ Bengaluru    │ Nomad Stores          │         17120 │     3 │
│ Bhopal       │ Leader                │      16529970 │     1 │
│ Bhopal       │ Epic Stores           │      10237161 │     2 │
│ Bhopal       │ Nomad Stores          │       9654955 │     3 │
│ Bhubaneshwar │ Info Stores           │        893857 │     1 │
│ Chennai      │ Surge Stores          │      18227503 │     1 │
│ Delhi NCR    │ Electric

Next Window Function (Very Important)

Now we move to LEAD().

This unlocks powerful analyses like:

time between purchases
customer churn detection
next purchase prediction
gap analysis
retention analysis

Example business question we will solve:

Find customers whose next purchase happened after more than 30 days.

This is a classic interview SQL problem.

Now Let's Solve a Real Business Problem
Business Question

**Find customers whose next purchase happened after more than 30 days.**

This is common in:

churn detection

customer lifecycle analysis

marketing pipelines

In [94]:
con.sql(
    """
    WITH purchase_gaps AS (
    SELECT
      customer_code,
      order_date,

      LEAD(order_date) OVER(
        PARTITION BY customer_code
        ORDER BY order_date
      ) AS next_order_date
    FROM transactions
    )

    SELECT
      customer_code,
      order_date,
      next_order_date,
      DATEDIFF('day', order_date, next_order_date) AS gap_days
    FROM purchase_gaps
    WHERE DATEDIFF('day', order_date, next_order_date) > 30
    ORDER BY gap_days
    """
)

┌───────────────┬─────────────────────┬─────────────────────┬──────────┐
│ customer_code │     order_date      │   next_order_date   │ gap_days │
│    varchar    │    timestamp_ns     │    timestamp_ns     │  int64   │
├───────────────┼─────────────────────┼─────────────────────┼──────────┤
│ Cus034        │ 2019-12-09 00:00:00 │ 2020-01-10 00:00:00 │       32 │
│ Cus031        │ 2019-04-05 00:00:00 │ 2019-05-07 00:00:00 │       32 │
│ Cus009        │ 2020-04-29 00:00:00 │ 2020-06-03 00:00:00 │       35 │
│ Cus028        │ 2017-10-16 00:00:00 │ 2017-11-20 00:00:00 │       35 │
│ Cus034        │ 2018-12-03 00:00:00 │ 2019-01-08 00:00:00 │       36 │
│ Cus034        │ 2019-10-17 00:00:00 │ 2019-11-29 00:00:00 │       43 │
│ Cus034        │ 2019-02-28 00:00:00 │ 2019-05-10 00:00:00 │       71 │
└───────────────┴─────────────────────┴─────────────────────┴──────────┘

Business Problem

Find customers whose next purchase happened in a different market than their previous purchase.

In [95]:
con.sql(
    """
    WITH new_market AS(
      SELECT
        market_code,
        order_date,
        customer_code,

        LEAD(market_code) OVER(
          PARTITION BY customer_code
          ORDER BY order_date
        ) AS next_market
      FROM transactions

    )
  SELECT
    customer_code,
    market_code,
    next_market,
    order_date
    FROM new_market
    WHERE market_code <> next_market
    ORDER BY customer_code, order_date
    LIMIT 20

    """
)

┌───────────────┬─────────────┬─────────────┬─────────────────────┐
│ customer_code │ market_code │ next_market │     order_date      │
│    varchar    │   varchar   │   varchar   │    timestamp_ns     │
├───────────────┼─────────────┼─────────────┼─────────────────────┤
│ Cus001        │ Mark010     │ Mark001     │ 2017-10-09 00:00:00 │
│ Cus001        │ Mark010     │ Mark001     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark010     │ Mark001     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark001     │ Mark010     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark010     │ Mark001     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark001     │ Mark010     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark001     │ Mark010     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark001     │ Mark010     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark010     │ Mark001     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark001     │ Mark010     │ 2017-10-10 00:00:00 │
│ Cus001        │ Mark010     │ Mark001     │ 20

LEAD() USING JOINS

In [96]:
con.sql(
    """
    WITH customer_journey AS (
    SELECT
      t.customer_code,
      c.customer_name,
      t.market_code,
      m.markets_name AS current_market,
      t.order_date,

    LEAD(t.market_code) OVER(
        PARTITION BY t.customer_code
        ORDER BY t.order_date
    ) AS next_market_code

  FROM transactions t
  JOIN customers c
    ON t.customer_code = c.customer_code
  JOIN markets m
    ON t.market_code = m.market_code
)

  SELECT
    cj.customer_name,
    cj.current_market,
    m2.markets_name AS next_market,
    cj.order_date

  FROM customer_journey cj
  JOIN markets m2
    ON cj.next_market_code = m2.market_code

  WHERE cj.market_code <> cj.next_market_code
  ORDER BY cj.customer_name, cj.order_date;
    """
)

┌──────────────────┬────────────────┬──────────────┬─────────────────────┐
│  customer_name   │ current_market │ next_market  │     order_date      │
│     varchar      │    varchar     │   varchar    │    timestamp_ns     │
├──────────────────┼────────────────┼──────────────┼─────────────────────┤
│ Acclaimed Stores │ Mumbai         │ Kanpur       │ 2017-10-09 00:00:00 │
│ Acclaimed Stores │ Kanpur         │ Mumbai       │ 2017-10-10 00:00:00 │
│ Acclaimed Stores │ Mumbai         │ Kanpur       │ 2017-10-16 00:00:00 │
│ Acclaimed Stores │ Mumbai         │ Kanpur       │ 2017-10-17 00:00:00 │
│ Acclaimed Stores │ Kanpur         │ Mumbai       │ 2017-10-17 00:00:00 │
│ Acclaimed Stores │ Kanpur         │ Mumbai       │ 2017-10-17 00:00:00 │
│ Acclaimed Stores │ Mumbai         │ Kanpur       │ 2017-10-23 00:00:00 │
│ Acclaimed Stores │ Kanpur         │ Mumbai       │ 2017-10-24 00:00:00 │
│ Acclaimed Stores │ Mumbai         │ Kanpur       │ 2017-10-30 00:00:00 │
│ Acclaimed Stores │ Kanp

Business Problem

Find customers who purchased in three different markets consecutively.

In [97]:
con.sql(
    """
    WITH market_sequence AS (
SELECT
    customer_code,
    market_code,
    order_date,

    LAG(market_code) OVER(
        PARTITION BY customer_code
        ORDER BY order_date
    ) AS previous_market,

    LEAD(market_code) OVER(
        PARTITION BY customer_code
        ORDER BY order_date
    ) AS next_market

FROM transactions
)

SELECT *
FROM market_sequence
WHERE
    previous_market IS NOT NULL
    AND next_market IS NOT NULL
    AND previous_market <> market_code
    AND market_code <> next_market
    AND previous_market <> next_market
  ORDER BY customer_code, order_date
  LIMIT 20;
    """
)

┌───────────────┬─────────────┬─────────────────────┬─────────────────┬─────────────┐
│ customer_code │ market_code │     order_date      │ previous_market │ next_market │
│    varchar    │   varchar   │    timestamp_ns     │     varchar     │   varchar   │
├───────────────┼─────────────┼─────────────────────┼─────────────────┼─────────────┤
│ Cus002        │ Mark002     │ 2017-10-10 00:00:00 │ Mark005         │ Mark007     │
│ Cus002        │ Mark007     │ 2017-10-10 00:00:00 │ Mark002         │ Mark005     │
│ Cus002        │ Mark007     │ 2017-10-10 00:00:00 │ Mark005         │ Mark002     │
│ Cus002        │ Mark007     │ 2017-10-18 00:00:00 │ Mark002         │ Mark006     │
│ Cus002        │ Mark006     │ 2017-10-19 00:00:00 │ Mark007         │ Mark005     │
│ Cus002        │ Mark007     │ 2017-10-27 00:00:00 │ Mark005         │ Mark002     │
│ Cus002        │ Mark002     │ 2017-10-31 00:00:00 │ Mark007         │ Mark005     │
│ Cus002        │ Mark002     │ 2017-11-21 00:00:00 │ 

**MODULE 3 — Data Cleaning & Transformation SQL**

Data engineering pipelines rarely receive perfectly clean data.
Raw datasets often contain:

missing values

inconsistent formats

malformed strings

incorrect types

duplicate records

This module focuses on transforming raw data into clean analytical datasets.

Typical usage in pipelines:

Raw Data (Bronze)
        ↓
Cleaning & Normalization
        ↓
Analytics Ready Tables (Silver / Gold)

**Sub-Module 3.1 — NULL Handling**
Concept

NULL represents missing or unknown data.

Important properties:

NULL ≠ 0
NULL ≠ empty string
NULL ≠ FALSE

NULL participates in three-valued logic:

Expression	Result
TRUE AND NULL	NULL
FALSE AND NULL	FALSE
NULL = NULL	NULL

Because of this behavior, NULL must be handled explicitly during transformations.

**Function 1 — COALESCE()**
Purpose

Returns the first non-NULL value in a list of expressions.

Syntax:

COALESCE(value1, value2, value3, ...)

Execution logic:

check value1
if NULL → check value2
if NULL → check value3
return first non-NULL value

In [98]:
con.sql(
    """
    SELECT
      customer_code,
      product_code,
      market_code,
      order_date,
      sales_amount,
      COALESCE(sales_amount, 0) AS cleaned_sales
    FROM transactions
    ORDER BY customer_code, order_date;
    """
)

┌───────────────┬──────────────┬─────────────┬─────────────────────┬──────────────┬───────────────┐
│ customer_code │ product_code │ market_code │     order_date      │ sales_amount │ cleaned_sales │
│    varchar    │   varchar    │   varchar   │    timestamp_ns     │    int64     │     int64     │
├───────────────┼──────────────┼─────────────┼─────────────────────┼──────────────┼───────────────┤
│ Cus001        │ Prod093      │ Mark010     │ 2017-10-04 00:00:00 │         1093 │          1093 │
│ Cus001        │ Prod105      │ Mark010     │ 2017-10-04 00:00:00 │          954 │           954 │
│ Cus001        │ Prod260      │ Mark010     │ 2017-10-04 00:00:00 │          560 │           560 │
│ Cus001        │ Prod090      │ Mark010     │ 2017-10-05 00:00:00 │          468 │           468 │
│ Cus001        │ Prod093      │ Mark010     │ 2017-10-05 00:00:00 │         5171 │          5171 │
│ Cus001        │ Prod105      │ Mark010     │ 2017-10-05 00:00:00 │         5866 │          5866 │


Another Real Query (More Practical)

Let’s clean revenue before aggregation.

Business question:

Total revenue per market but treat missing sales as 0.

Next Function (Sub-Module 3.1)
NULLIF()

This function is often used to convert bad values into NULLs.

Example scenario:

source system stores missing values as:
0
''
'N/A'

We convert them into proper SQL NULL values.

Example:

NULLIF(column,0)

Meaning:

if column = 0 → return NULL
else → return column

In [99]:
con.sql(
    """
    SELECT
      market_code,
      SUM(COALESCE(sales_amount,0)) AS total_revenue
    FROM transactions
    GROUP BY market_code
    ORDER BY total_revenue DESC;
    """
)

┌─────────────┬───────────────┐
│ market_code │ total_revenue │
│   varchar   │    int128     │
├─────────────┼───────────────┤
│ Mark004     │     520721134 │
│ Mark002     │     150180636 │
│ Mark003     │     132526737 │
│ Mark011     │      55026321 │
│ Mark007     │      42128765 │
│ Mark010     │      18813466 │
│ Mark001     │      18227503 │
│ Mark013     │      16525290 │
│ Mark005     │      13583923 │
│ Mark014     │       7436823 │
│ Mark009     │       4428393 │
│ Mark008     │       3094007 │
│ Mark012     │       2605796 │
│ Mark015     │        893857 │
│ Mark006     │        373115 │
├─────────────┴───────────────┤
│ 15 rows           2 columns │
└─────────────────────────────┘

In [100]:
con.sql(
    """
    SELECT
      customer_code,
      product_code,
      NULLIF(sales_amount, 0) AS normalized_sales
    FROM transactions
    LIMIT 25;
    """
)

┌───────────────┬──────────────┬──────────────────┐
│ customer_code │ product_code │ normalized_sales │
│    varchar    │   varchar    │      int64       │
├───────────────┼──────────────┼──────────────────┤
│ Cus001        │ Prod001      │            41241 │
│ Cus002        │ Prod001      │               -1 │
│ Cus003        │ Prod002      │              875 │
│ Cus003        │ Prod002      │              583 │
│ Cus004        │ Prod002      │             7176 │
│ Cus005        │ Prod003      │              500 │
│ Cus005        │ Prod003      │              250 │
│ Cus005        │ Prod003      │            21412 │
│ Cus005        │ Prod003      │            19213 │
│ Cus005        │ Prod003      │           170185 │
│   ·           │    ·         │              ·   │
│   ·           │    ·         │              ·   │
│   ·           │    ·         │              ·   │
│ Cus006        │ Prod003      │            30306 │
│ Cus006        │ Prod005      │            52319 │
│ Cus006    

In [101]:
con.sql(
    """
    SELECT
      customer_code,
      product_code,
      COALESCE(NULLIF(sales_amount,0), NULL)
    FROM transactions
    LIMIT 25;
    """
)

┌───────────────┬──────────────┬───────────────────────────────────────────┐
│ customer_code │ product_code │ COALESCE("nullif"(sales_amount, 0), NULL) │
│    varchar    │   varchar    │                   int64                   │
├───────────────┼──────────────┼───────────────────────────────────────────┤
│ Cus001        │ Prod001      │                                     41241 │
│ Cus002        │ Prod001      │                                        -1 │
│ Cus003        │ Prod002      │                                       875 │
│ Cus003        │ Prod002      │                                       583 │
│ Cus004        │ Prod002      │                                      7176 │
│ Cus005        │ Prod003      │                                       500 │
│ Cus005        │ Prod003      │                                       250 │
│ Cus005        │ Prod003      │                                     21412 │
│ Cus005        │ Prod003      │                                     19213 │

**Detect Bad Numeric Values**

Before applying:

NULLIF()
COALESCE()
CAST()

we first check if they are needed.

Detection Query 1 — Check for NULL Values

In [102]:
con.sql(
    """
    SELECT
      COUNT(*) AS total_rows,
      COUNT(sales_amount) AS non_null_sales,
      COUNT(*) - COUNT(sales_amount) AS null_sales
    FROM transactions
    """
)

┌────────────┬────────────────┬────────────┐
│ total_rows │ non_null_sales │ null_sales │
│   int64    │     int64      │   int64    │
├────────────┼────────────────┼────────────┤
│     150283 │         150283 │          0 │
└────────────┴────────────────┴────────────┘

Detection Query 2 — Detect Placeholder Values

Check if 0 is being used as a placeholder.

If this returns many rows, we investigate whether 0 means missing data.

In [103]:
con.sql(
    """
    SELECT
      COUNT(*) AS zero_sales_records
    FROM transactions
    WHERE sales_amount = 0;
    """
)

┌────────────────────┐
│ zero_sales_records │
│       int64        │
├────────────────────┤
│               1609 │
└────────────────────┘

**Sub-Module 3.1 — NULL Handling**
NULLIF()
Concept

NULLIF() is used to convert specific values into NULL.

It compares two expressions:

NULLIF(value, comparison_value)

If both values are equal, the function returns NULL.
Otherwise, it returns the original value.

Example
NULLIF(0,0) → NULL
NULLIF(5,0) → 5

Why It Is Used in Data Engineering

Many raw data sources store missing values using placeholders instead of proper SQL NULL.

Common placeholders:

Raw Value	Meaning
0	missing value
-1	missing value
''	empty value
'N/A'	unavailable

NULLIF() converts these placeholder values into proper SQL NULLs so that the data behaves correctly in:

aggregations

joins

transformations

analytics queries

For analytics we want:

**invalid values → converted to NULL**

**missing numeric values → replaced with safe defaults**

**readable dimension attributes**

**Query — Cleaning Sales Data with Joins**

In [104]:
con.sql(
    """
    SELECT
      t.customer_code,
      c.customer_name,
      t.product_code,
      p.product_type,
      t.market_code,
      m.markets_name,
      t.order_date,
      -- convert placeholder sales values to NULL
      NULLIF(t.sales_amount, 0) AS nomalized_sales,

      -- replace NULL values with 0 for safe analytics
      COALESCE(NULLIF(t.sales_amount, 0), 0) AS cleaned_sales


    FROM transactions t

    LEFT JOIN customers c
      ON t.customer_code = c.customer_code

    LEFT JOIN products p
      ON t.product_code = p.product_code

    LEFT JOIN markets m
      ON t.market_code = m.market_code

    ORDER BY t.customer_code, t.order_date
    """
)

┌───────────────┬───────────────┬──────────────┬──────────────┬─────────────┬──────────────┬─────────────────────┬─────────────────┬───────────────┐
│ customer_code │ customer_name │ product_code │ product_type │ market_code │ markets_name │     order_date      │ nomalized_sales │ cleaned_sales │
│    varchar    │    varchar    │   varchar    │   varchar    │   varchar   │   varchar    │    timestamp_ns     │      int64      │     int64     │
├───────────────┼───────────────┼──────────────┼──────────────┼─────────────┼──────────────┼─────────────────────┼─────────────────┼───────────────┤
│ Cus001        │ Surge Stores  │ Prod093      │ Own Brand    │ Mark010     │ Kochi        │ 2017-10-04 00:00:00 │            1093 │          1093 │
│ Cus001        │ Surge Stores  │ Prod105      │ Own Brand    │ Mark010     │ Kochi        │ 2017-10-04 00:00:00 │             954 │           954 │
│ Cus001        │ Surge Stores  │ Prod260      │ Own Brand    │ Mark010     │ Kochi        │ 2017-10-04 00

What This Query Is Doing
1️⃣ Join Fact Table with Dimension Tables
transactions (fact)
       ↓
customers
products
markets

This enriches the transaction records with descriptive attributes.

Result columns include:

customer_name
product_type
markets_name
2️⃣ Normalize Placeholder Sales Values
NULLIF(t.sales_amount, 0)

Logic:

sales_amount	normalized_sales
500	500
0	NULL
300	300

Here we assume:

0 = missing or invalid sales value

So we convert it into proper SQL NULL.

3️⃣ Replace NULL Values for Calculations
COALESCE(NULLIF(t.sales_amount,0),0)

Final Data Behavior
sales_amount	normalized_sales	cleaned_sales

500	               500	           500

0	                 NULL	            0

NULL	             NULL	            0

Meaning:

normalized_sales → represents real data state
cleaned_sales → safe value for analytics calculations

**Sub-Module 3.2 — Data Type Normalization**

Concept

Raw ingestion systems often store values in the wrong data type.

Common problems in raw datasets:

Raw Value	Problem

'500'	number stored as string

'2024-01-01'	date stored as text

'1,200'	formatted numbers

'$300'	currency symbols

Before performing analytics, these values must be converted into proper SQL data types.

Typical conversion functions:

CAST()
TRY_CAST() / SAFE_CAST()

Purpose:

string → numeric

string → date

string → timestamp

This step usually happens in the Silver layer transformation of a data pipeline.

**Real Business Problem**

Suppose the ingestion pipeline stores sales_amount as text instead of numeric.

Before calculating revenue, we must:

**Clean placeholder values**

**Convert text to numeric**

**Handle invalid values safely**

**Query — Normalize Data Types with Cleaning**

In [105]:
con.sql(
    """
    SELECT
      t.customer_code,
      c.customer_name,
      t.product_code,
      p.product_type,
      t.market_code,
      m.markets_name,
      t.order_date,

      -- normalize placeholder values
      NULLIF(t.sales_amount, 0) AS normalized_sales,

      -- convert sales to numeric
      CAST(COALESCE(NULLIF(t.sales_amount,0),0) AS DOUBLE) AS numeric_sales

  FROM transactions t

  LEFT JOIN customers c
    ON t.customer_code = c.customer_code

  LEFT JOIN products p
    ON t.product_code = p.product_code

  LEFT JOIN markets m
    ON t.market_code = m.market_code

  ORDER BY t.customer_code, t.order_date;
    """
)

┌───────────────┬───────────────┬──────────────┬──────────────┬─────────────┬──────────────┬─────────────────────┬──────────────────┬───────────────┐
│ customer_code │ customer_name │ product_code │ product_type │ market_code │ markets_name │     order_date      │ normalized_sales │ numeric_sales │
│    varchar    │    varchar    │   varchar    │   varchar    │   varchar   │   varchar    │    timestamp_ns     │      int64       │    double     │
├───────────────┼───────────────┼──────────────┼──────────────┼─────────────┼──────────────┼─────────────────────┼──────────────────┼───────────────┤
│ Cus001        │ Surge Stores  │ Prod093      │ Own Brand    │ Mark010     │ Kochi        │ 2017-10-04 00:00:00 │             1093 │        1093.0 │
│ Cus001        │ Surge Stores  │ Prod105      │ Own Brand    │ Mark010     │ Kochi        │ 2017-10-04 00:00:00 │              954 │         954.0 │
│ Cus001        │ Surge Stores  │ Prod260      │ Own Brand    │ Mark010     │ Kochi        │ 2017-10

Now Continuing Module 3 (Detection First)

Next we detect string inconsistencies in the dataset.

Run this profiling query:

In [106]:
con.sql(
    """
    SELECT DISTINCT markets_name
    FROM markets
    ORDER BY markets_name
    """
)

┌──────────────┐
│ markets_name │
│   varchar    │
├──────────────┤
│ Ahmedabad    │
│ Bengaluru    │
│ Bhopal       │
│ Bhubaneshwar │
│ Chennai      │
│ Delhi NCR    │
│ Hyderabad    │
│ Kanpur       │
│ Kochi        │
│ Lucknow      │
│ Mumbai       │
│ Nagpur       │
│ New York     │
│ Paris        │
│ Patna        │
│ Surat        │
├──────────────┤
│   16 rows    │
└──────────────┘

Next Detection Step (More Interesting)

Markets are clean. Now let's check customer data, which is usually messier in real systems.

In [107]:
con.sql(
    """
    SELECT
      DISTINCT customer_code
    FROM customers
    ORDER BY customer_code;
    """
)

┌───────────────┐
│ customer_code │
│    varchar    │
├───────────────┤
│ Cus001        │
│ Cus002        │
│ Cus003        │
│ Cus004        │
│ Cus005        │
│ Cus006        │
│ Cus007        │
│ Cus008        │
│ Cus009        │
│ Cus010        │
│   ·           │
│   ·           │
│   ·           │
│ Cus029        │
│ Cus030        │
│ Cus031        │
│ Cus032        │
│ Cus033        │
│ Cus034        │
│ Cus035        │
│ Cus036        │
│ Cus037        │
│ Cus038        │
├───────────────┤
│    38 rows    │
│  (20 shown)   │
└───────────────┘

In [108]:
con.sql(
    """
    SELECT
      COUNT(*) AS total_rows,
      COUNT(DISTINCT customer_code) AS unique_customers
    FROM customers;
    """
)

┌────────────┬──────────────────┐
│ total_rows │ unique_customers │
│   int64    │      int64       │
├────────────┼──────────────────┤
│         38 │               38 │
└────────────┴──────────────────┘

Next Topic — REGEXP (Pattern-Based Data Cleaning)

Now we move to a more powerful transformation tool.

Concept

REGEXP functions allow SQL to detect and manipulate text using patterns.

Used for:

validating formats

extracting information from strings

detecting malformed values

Common functions:

REGEXP_EXTRACT
REGEXP_REPLACE
REGEXP_MATCHES
Why Data Engineers Use REGEXP

Raw data often contains inconsistent formats:

Raw Value	Problem

"Order-12345"	contains prefix

"Cust#C012"	unwanted characters

"sales: 500"	mixed text and numbers

Regex allows us to extract the meaningful value.

**Real Example with Our Dataset**

Suppose we want to extract only numeric characters from product_code.

Example values:

Prod384

Prod129

Prod88

We extract the numeric part.

In [109]:
con.sql(
    """
    SELECT
      product_code,
      REGEXP_EXTRACT(product_code,'[0-9]+') AS product_number
    FROM transactions
    LIMIT 20
    """
)

┌──────────────┬────────────────┐
│ product_code │ product_number │
│   varchar    │    varchar     │
├──────────────┼────────────────┤
│ Prod001      │ 001            │
│ Prod001      │ 001            │
│ Prod002      │ 002            │
│ Prod002      │ 002            │
│ Prod002      │ 002            │
│ Prod003      │ 003            │
│ Prod003      │ 003            │
│ Prod003      │ 003            │
│ Prod003      │ 003            │
│ Prod003      │ 003            │
│ Prod003      │ 003            │
│ Prod003      │ 003            │
│ Prod004      │ 004            │
│ Prod004      │ 004            │
│ Prod005      │ 005            │
│ Prod003      │ 003            │
│ Prod005      │ 005            │
│ Prod005      │ 005            │
│ Prod005      │ 005            │
│ Prod005      │ 005            │
├──────────────┴────────────────┤
│ 20 rows             2 columns │
└───────────────────────────────┘

In [110]:
con.sql(
    """
    SELECT DISTINCT product_code
    FROM transactions
    LIMIT 20;
    """
)

┌──────────────┐
│ product_code │
│   varchar    │
├──────────────┤
│ Prod299      │
│ Prod303      │
│ Prod305      │
│ Prod307      │
│ Prod308      │
│ Prod309      │
│ Prod310      │
│ Prod311      │
│ Prod312      │
│ Prod316      │
│ Prod318      │
│ Prod319      │
│ Prod321      │
│ Prod324      │
│ Prod325      │
│ Prod327      │
│ Prod328      │
│ Prod330      │
│ Prod333      │
│ Prod334      │
├──────────────┤
│   20 rows    │
└──────────────┘

Production Transformation Pattern

Often pipelines combine REGEXP + CAST.

Example:

In [111]:
con.sql("""SELECT
    product_code,
    CAST(REGEXP_EXTRACT(product_code,'[0-9]+') AS INTEGER) AS product_id
FROM transactions;
""")

┌──────────────┬────────────┐
│ product_code │ product_id │
│   varchar    │   int32    │
├──────────────┼────────────┤
│ Prod001      │          1 │
│ Prod001      │          1 │
│ Prod002      │          2 │
│ Prod002      │          2 │
│ Prod002      │          2 │
│ Prod003      │          3 │
│ Prod003      │          3 │
│ Prod003      │          3 │
│ Prod003      │          3 │
│ Prod003      │          3 │
│    ·         │          · │
│    ·         │          · │
│    ·         │          · │
│ Prod054      │         54 │
│ Prod054      │         54 │
│ Prod054      │         54 │
│ Prod054      │         54 │
│ Prod054      │         54 │
│ Prod054      │         54 │
│ Prod054      │         54 │
│ Prod054      │         54 │
│ Prod054      │         54 │
│ Prod054      │         54 │
├──────────────┴────────────┤
│ ? rows          2 columns │
└───────────────────────────┘

Now Let's Build a Real Transformation Step

We will simulate a Silver Layer transformation for analytics.

Goal

Create a cleaned transaction dataset containing:

customer_name

market_name

product_id

clean_sales

order_date

Problems we fix:

Problem	Solution

sales placeholder values	NULLIF()

safe numeric values	COALESCE()

product numeric ID	REGEXP_EXTRACT()

dimension lookup	JOIN

Correct Query to Detect Duplicate product_code

In [112]:
con.sql(
    """
    SELECT
      product_code,
        COUNT(*) AS occurences
      FROM products
      GROUP BY product_code
      HAVING COUNT(*) > 1
    """
)

┌──────────────┬────────────┐
│ product_code │ occurences │
│   varchar    │   int64    │
├──────────────┴────────────┤
│          0 rows           │
└───────────────────────────┘

When We Use ROW_NUMBER()

Once duplicates are confirmed, we resolve them:

In [113]:
con.sql(
    """
    SELECT *
    FROM (
      SELECT
        *,
        ROW_NUMBER() OVER(
          PARTITION BY product_code
          ORDER BY product_code
        )AS rn
      FROM products
    )t
    WHERE rn = 1
    """
)

┌──────────────┬──────────────┬───────┐
│ product_code │ product_type │  rn   │
│   varchar    │   varchar    │ int64 │
├──────────────┼──────────────┼───────┤
│ Prod068      │ Distribution │     1 │
│ Prod074      │ Distribution │     1 │
│ Prod083      │ Own Brand    │     1 │
│ Prod097      │ Own Brand    │     1 │
│ Prod114      │ Distribution │     1 │
│ Prod124      │ Own Brand    │     1 │
│ Prod142      │ Own Brand    │     1 │
│ Prod144      │ Distribution │     1 │
│ Prod186      │ Own Brand    │     1 │
│ Prod191      │ Distribution │     1 │
│    ·         │     ·        │     · │
│    ·         │     ·        │     · │
│    ·         │     ·        │     · │
│ Prod141      │ Own Brand    │     1 │
│ Prod148      │ Own Brand    │     1 │
│ Prod150      │ Own Brand    │     1 │
│ Prod175      │ Distribution │     1 │
│ Prod183      │ Own Brand    │     1 │
│ Prod188      │ Distribution │     1 │
│ Prod205      │ Distribution │     1 │
│ Prod246      │ Own Brand    │     1 │


Query 1 — Data Cleaning Pipeline

This query simulates a Silver layer transformation.

In [114]:
con.sql(
    """
    SELECT
      t.customer_code,
      c.customer_name,

      t.market_code,
      m.markets_name AS market_name,

      t.product_code,

      -- extract numeric product id
      CAST(REGEXP_EXTRACT(t.product_code,'[0-9]+') AS INTEGER) AS product_id,

      -- normalize placeholder values
      NULLIF(t.sales_amount,0) AS normalized_sales,

      -- safe sales value for analytics
      COALESCE(NULLIF(t.sales_amount,0),0) AS clean_sales,

      t.order_date,

      -- normalize time granularity
      DATE_TRUNC('month', t.order_date) AS sales_month

  FROM transactions t

  LEFT JOIN customers c
    ON t.customer_code = c.customer_code

  LEFT JOIN markets m
    ON t.market_code = m.market_code

  ORDER BY t.order_date
  LIMIT 50;

    """
)

┌───────────────┬────────────────────┬─────────────┬─────────────┬──────────────┬────────────┬──────────────────┬─────────────┬─────────────────────┬─────────────┐
│ customer_code │   customer_name    │ market_code │ market_name │ product_code │ product_id │ normalized_sales │ clean_sales │     order_date      │ sales_month │
│    varchar    │      varchar       │   varchar   │   varchar   │   varchar    │   int32    │      int64       │    int64    │    timestamp_ns     │    date     │
├───────────────┼────────────────────┼─────────────┼─────────────┼──────────────┼────────────┼──────────────────┼─────────────┼─────────────────────┼─────────────┤
│ Cus010        │ Atlas Stores       │ Mark003     │ Ahmedabad   │ Prod188      │        188 │             1435 │        1435 │ 2017-10-04 00:00:00 │ 2017-10-01  │
│ Cus035        │ Relief             │ Mark007     │ Bhopal      │ Prod093      │         93 │              440 │         440 │ 2017-10-04 00:00:00 │ 2017-10-01  │
│ Cus012        

Query 2 — Analytics-Ready Aggregation

Now we use the cleaned data for analysis.

In [115]:
con.sql(
    """
    SELECT
      m.markets_name AS market,
      DATE_TRUNC('month', t.order_date) AS sales_month,

      SUM(COALESCE(NULLIF(t.sales_amount,0),0)) AS total_revenue,

      COUNT(DISTINCT t.customer_code) AS unique_customers,

      COUNT(*) AS total_transactions

    FROM transactions t

  LEFT JOIN markets m
    ON t.market_code = m.market_code

  GROUP BY
    market,
    sales_month

  ORDER BY
    sales_month,
    total_revenue DESC;
    """
)

┌──────────────┬─────────────┬───────────────┬──────────────────┬────────────────────┐
│    market    │ sales_month │ total_revenue │ unique_customers │ total_transactions │
│   varchar    │    date     │    int128     │      int64       │       int64        │
├──────────────┼─────────────┼───────────────┼──────────────────┼────────────────────┤
│ Delhi NCR    │ 2017-10-01  │      13067710 │                4 │               1545 │
│ Mumbai       │ 2017-10-01  │       4048360 │               17 │                386 │
│ Ahmedabad    │ 2017-10-01  │       3509040 │               11 │                529 │
│ Bhopal       │ 2017-10-01  │       1685300 │               12 │                290 │
│ Chennai      │ 2017-10-01  │       1062702 │                1 │                 35 │
│ Nagpur       │ 2017-10-01  │        971566 │               14 │               1014 │
│ Kochi        │ 2017-10-01  │        764470 │                3 │                215 │
│ Kanpur       │ 2017-10-01  │        50878

Practice Query — Module 3 + Aggregation
Business Question

Find the top 5 markets by total revenue for each year.

Rules:

1️⃣ Use DATE_TRUNC('year', order_date) to create the year.

2️⃣ Ignore placeholder sales values (sales_amount = 0).

3️⃣ Use markets table to display market name instead of market_code.

4️⃣ Show:

year

market_name

total_revenue

total_transactions

unique_customers

5️⃣ Sort the result by:

year
total_revenue DESC

6️⃣ Return only top 5 markets per year.

In [116]:
con.sql(
    """
    WITH market_year_sales AS(
      SELECT
        m.markets_name AS market_name,

        DATE_TRUNC('year', t.order_date) AS sales_year,

        SUM(COALESCE(NULLIF(t.sales_amount,0),0)) AS total_revenue,

        COUNT(DISTINCT t.customer_code) AS unique_customers,

        COUNT(*) AS total_transactions

      FROM transactions t
      LEFT JOIN markets m
        ON t.market_code = m.market_code

      GROUP BY
        market_name,
        sales_year
  ),
  ranked_market AS(
    SELECT
      *,
      ROW_NUMBER() OVER(
        PARTITION BY sales_year
        ORDER BY total_revenue DESC
      )AS revenue_rank
    FROM market_year_sales
  )
  SELECT *
  FROM ranked_market
  WHERE revenue_rank <= 5
  ORDER BY sales_year, revenue_rank

    """
)

┌─────────────┬────────────┬───────────────┬──────────────────┬────────────────────┬──────────────┐
│ market_name │ sales_year │ total_revenue │ unique_customers │ total_transactions │ revenue_rank │
│   varchar   │    date    │    int128     │      int64       │       int64        │    int64     │
├─────────────┼────────────┼───────────────┼──────────────────┼────────────────────┼──────────────┤
│ Delhi NCR   │ 2017-01-01 │      49960727 │                4 │               5222 │            1 │
│ Mumbai      │ 2017-01-01 │      14916377 │               21 │               1158 │            2 │
│ Ahmedabad   │ 2017-01-01 │      11442751 │               11 │               1893 │            3 │
│ Bhopal      │ 2017-01-01 │       5566887 │               13 │               1024 │            4 │
│ Nagpur      │ 2017-01-01 │       3979300 │               16 │               3782 │            5 │
│ Delhi NCR   │ 2018-01-01 │     221263704 │                4 │              20503 │            1 │


Business Question

Find the top 3 customers by revenue inside each market per year.

Requirements:

Use tables:

transactions
customers
markets

Steps you should follow:

1️⃣ Join the tables

2️⃣ Aggregate revenue per

market
year
customer

3️⃣ Rank customers using

ROW_NUMBER()
PARTITION BY market, year
ORDER BY revenue DESC

4️⃣ Keep

top 3 customers per market per year

In [117]:
con.sql(
    """
    WITH market_year_sales AS(
      SELECT
        m.markets_name AS market_name,
        c.customer_name,

        DATE_TRUNC('year', t.order_date) AS sales_year,

        SUM(COALESCE(NULLIF(t.sales_amount,0),0)) AS total_revenue,

        COUNT(*) AS total_transactions

      FROM transactions t
      LEFT JOIN markets m
        ON t.market_code = m.market_code

      LEFT JOIN customers c
        ON t.customer_code = c.customer_code

      GROUP BY
        market_name,
        sales_year,
        customer_name
    ),
    rank_customer AS(
      SELECT
        *,
        ROW_NUMBER() OVER(
          PARTITION BY market_name, sales_year
          ORDER BY total_revenue DESC
        ) AS customer_rank
      FROM market_year_sales
    )
    SELECT *
    FROM rank_customer
    WHERE customer_rank <= 3
    ORDER BY sales_year, customer_rank, market_name
    """
)

┌──────────────┬───────────────────────┬────────────┬───────────────┬────────────────────┬───────────────┐
│ market_name  │     customer_name     │ sales_year │ total_revenue │ total_transactions │ customer_rank │
│   varchar    │        varchar        │    date    │    int128     │       int64        │     int64     │
├──────────────┼───────────────────────┼────────────┼───────────────┼────────────────────┼───────────────┤
│ Ahmedabad    │ Excel Stores          │ 2017-01-01 │       2659397 │                687 │             1 │
│ Bengaluru    │ Nomad Stores          │ 2017-01-01 │             0 │                  5 │             1 │
│ Bhopal       │ Synthetic             │ 2017-01-01 │       1239176 │                 54 │             1 │
│ Bhubaneshwar │ Info Stores           │ 2017-01-01 │         96607 │                 22 │             1 │
│ Chennai      │ Surge Stores          │ 2017-01-01 │       1670830 │                 80 │             1 │
│ Delhi NCR    │ Electricalsara Store

In [118]:
con.sql(
    """
    SELECT
      m.markets_name,
      DATE_TRUNC('year', t.order_date) AS sales_year,
      COUNT(DISTINCT t.customer_code) AS customers
    FROM transactions t
    JOIN markets m
      ON t.market_code = m.market_code
    GROUP BY
      m.markets_name,
      sales_year
    ORDER BY customers DESC;
    """
)

┌──────────────┬────────────┬───────────┐
│ markets_name │ sales_year │ customers │
│   varchar    │    date    │   int64   │
├──────────────┼────────────┼───────────┤
│ Mumbai       │ 2019-01-01 │        23 │
│ Mumbai       │ 2018-01-01 │        22 │
│ Bengaluru    │ 2018-01-01 │        22 │
│ Mumbai       │ 2017-01-01 │        21 │
│ Nagpur       │ 2019-01-01 │        19 │
│ Mumbai       │ 2020-01-01 │        19 │
│ Nagpur       │ 2018-01-01 │        18 │
│ Nagpur       │ 2020-01-01 │        17 │
│ Nagpur       │ 2017-01-01 │        16 │
│ Ahmedabad    │ 2018-01-01 │        14 │
│    ·         │     ·      │         · │
│    ·         │     ·      │         · │
│    ·         │     ·      │         · │
│ Chennai      │ 2017-01-01 │         1 │
│ Lucknow      │ 2019-01-01 │         1 │
│ Bhubaneshwar │ 2019-01-01 │         1 │
│ Patna        │ 2019-01-01 │         1 │
│ Patna        │ 2017-01-01 │         1 │
│ Patna        │ 2020-01-01 │         1 │
│ Surat        │ 2017-01-01 │     

Business Problem

**Find the Top 3 Markets with the Largest Year-Over-Year Revenue Growth**

We want to answer:

Which markets improved the most compared to the previous year?

In [119]:
con.sql(
    """
    WITH yearly_market_sales AS (
      SELECT
        m.markets_name AS market_name,
        DATE_TRUNC('year', t.order_date) AS sales_year,

        SUM(COALESCE(NULLIF(t.sales_amount, 0), 0)) AS yearly_revenue

      FROM transactions t
      JOIN markets m
        ON t.market_code = m.market_code

      GROUP BY
        market_name,
        sales_year

    ),

    sales_with_growth AS(
      SELECT
        market_name,
        sales_year,
        yearly_revenue,

        LAG(yearly_revenue) OVER(
          PARTITION BY market_name
          ORDER BY sales_year
        )AS previous_year_revenue
      FROM yearly_market_sales
    ),

    growth_calculation AS (

  SELECT
    market_name,
    sales_year,
    yearly_revenue,
    previous_year_revenue,

    yearly_revenue - previous_year_revenue AS revenue_growth

  FROM sales_with_growth
),
ranked_growth AS (

  SELECT
    *,
    ROW_NUMBER() OVER(
        PARTITION BY sales_year
        ORDER BY revenue_growth DESC
    ) AS growth_rank

  FROM growth_calculation
)
SELECT *
  FROM ranked_growth
  WHERE growth_rank <= 3
  ORDER BY sales_year, growth_rank;
    """
)

┌──────────────┬────────────┬────────────────┬───────────────────────┬────────────────┬─────────────┐
│ market_name  │ sales_year │ yearly_revenue │ previous_year_revenue │ revenue_growth │ growth_rank │
│   varchar    │    date    │     int128     │        int128         │     int128     │    int64    │
├──────────────┼────────────┼────────────────┼───────────────────────┼────────────────┼─────────────┤
│ Lucknow      │ 2017-01-01 │         201186 │                  NULL │           NULL │           1 │
│ Nagpur       │ 2017-01-01 │        3979300 │                  NULL │           NULL │           2 │
│ Mumbai       │ 2017-01-01 │       14916377 │                  NULL │           NULL │           3 │
│ Delhi NCR    │ 2018-01-01 │      221263704 │              49960727 │      171302977 │           1 │
│ Mumbai       │ 2018-01-01 │       63073695 │              14916377 │       48157318 │           2 │
│ Ahmedabad    │ 2018-01-01 │       56001042 │              11442751 │       44558

**Rolling Time Windows**

Example business problem:

Find 7-day moving average revenue for each market

This uses:

**ROWS BETWEEN 6 PRECEDING AND CURRENT ROW**

In [120]:
con.sql(
    """
    WITH daily_market_sales AS(
      SELECT
        m.markets_name AS market_name,
        t.order_date,

        SUM(COALESCE(NULLIF(t.sales_amount,0), 0)) AS daily_revenue

      FROM transactions t
      LEFT JOIN markets m
        ON t.market_code = m.market_code

      GROUP BY
        market_name,
        order_date

    )

    SELECT
      market_name,
      order_date,
      daily_revenue,

      AVG(daily_revenue) OVER(
        PARTITION BY market_name
        ORDER BY order_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
      ) AS moving_avg_7_daily
    FROM daily_market_sales
    ORDER BY
      market_name,
      order_date
    LIMIT 25
    """
)

┌─────────────┬─────────────────────┬───────────────┬────────────────────┐
│ market_name │     order_date      │ daily_revenue │ moving_avg_7_daily │
│   varchar   │    timestamp_ns     │    int128     │       double       │
├─────────────┼─────────────────────┼───────────────┼────────────────────┤
│ Ahmedabad   │ 2017-10-04 00:00:00 │         47910 │            47910.0 │
│ Ahmedabad   │ 2017-10-05 00:00:00 │        168287 │           108098.5 │
│ Ahmedabad   │ 2017-10-06 00:00:00 │        422578 │           212925.0 │
│ Ahmedabad   │ 2017-10-09 00:00:00 │         30384 │          167289.75 │
│ Ahmedabad   │ 2017-10-10 00:00:00 │         13292 │           136490.2 │
│ Ahmedabad   │ 2017-10-11 00:00:00 │        185698 │           144691.5 │
│ Ahmedabad   │ 2017-10-12 00:00:00 │        262399 │ 161506.85714285713 │
│ Ahmedabad   │ 2017-10-13 00:00:00 │        234106 │  188106.2857142857 │
│ Ahmedabad   │ 2017-10-16 00:00:00 │        141735 │ 184313.14285714287 │
│ Ahmedabad   │ 2017-10-1

Deduplication Pipelines
Concept (Notebook Heading)

Deduplication removes duplicate records created by:

ingestion retries

upstream system bugs

CDC replays

late arriving events

Common strategy:

ROW_NUMBER()

PARTITION BY duplicate key

ORDER BY record priority

Query — Detect Duplicates First

Always detect before removing.

In [121]:
con.sql(
    """
    SELECT
      customer_code,
      order_date,
      product_code,

      COUNT(*) AS duplicated_count
    FROM transactions
    GROUP BY
      customer_code,
      product_code,
      order_date
    HAVING COUNT(*) > 1
    ORDER BY duplicated_count DESC
    """
)

┌───────────────┬─────────────────────┬──────────────┬──────────────────┐
│ customer_code │     order_date      │ product_code │ duplicated_count │
│    varchar    │    timestamp_ns     │   varchar    │      int64       │
├───────────────┼─────────────────────┼──────────────┼──────────────────┤
│ Cus003        │ 2018-06-25 00:00:00 │ Prod318      │                3 │
│ Cus003        │ 2018-09-24 00:00:00 │ Prod318      │                3 │
│ Cus003        │ 2017-10-23 00:00:00 │ Prod318      │                3 │
│ Cus003        │ 2018-02-12 00:00:00 │ Prod318      │                3 │
│ Cus003        │ 2018-03-19 00:00:00 │ Prod318      │                3 │
│ Cus022        │ 2017-12-07 00:00:00 │ Prod117      │                3 │
│ Cus022        │ 2018-08-09 00:00:00 │ Prod117      │                3 │
│ Cus002        │ 2018-01-05 00:00:00 │ Prod129      │                3 │
│ Cus016        │ 2017-11-17 00:00:00 │ Prod334      │                2 │
│ Cus003        │ 2017-11-20 00:00:00 

Query — Deduplicate Using ROW_NUMBER()

Now we remove duplicates.

In [122]:
con.sql(
    """
    WITH ranked_transactions AS (

    SELECT
      *,
      ROW_NUMBER() OVER(
        PARTITION BY
            customer_code,
            product_code,
            order_date
        ORDER BY
            order_date DESC
    ) AS rn

    FROM transactions

)

SELECT *
FROM ranked_transactions
WHERE rn = 1;

    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┬───────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │  rn   │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │ int64 │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┼───────┤
│ Prod040      │ Cus001        │ Mark001     │ 2019-07-11 00:00:00 │        40 │        25116 │ INR      │     1 │
│ Prod040      │ Cus001        │ Mark001     │ 2020-05-15 00:00:00 │        40 │        28042 │ INR      │     1 │
│ Prod090      │ Cus001        │ Mark010     │ 2017-10-31 00:00:00 │        10 │          671 │ INR      │     1 │
│ Prod090      │ Cus001        │ Mark010     │ 2017-11-30 00:00:00 │         7 │          421 │ INR      │     1 │
│ Prod090      │ Cus001        │ Mark010     │ 2018-01-31 00:00:00 │        33 │

What Changed in the Business Rule

New requirement:

Keep the latest transaction per:
customer_code
product_code
market_code

Tie-breaker:

If two rows have the same order_date
→ keep the one with higher sales_amount

In [123]:
con.sql(
    """
    WITH ranked_transactions AS (
      SELECT
        *,
        ROW_NUMBER() OVER(
          PARTITION BY
            customer_code,
            product_code,
            market_code
          ORDER BY
            order_date DESC,
            sales_amount DESC
        )AS rn
      FROM transactions
    )
    SELECT *
    FROM ranked_transactions
    WHERE rn = 1
    """
)

┌──────────────┬───────────────┬─────────────┬─────────────────────┬───────────┬──────────────┬──────────┬───────┐
│ product_code │ customer_code │ market_code │     order_date      │ sales_qty │ sales_amount │ currency │  rn   │
│   varchar    │    varchar    │   varchar   │    timestamp_ns     │   int64   │    int64     │ varchar  │ int64 │
├──────────────┼───────────────┼─────────────┼─────────────────────┼───────────┼──────────────┼──────────┼───────┤
│ Prod270      │ Cus021        │ Mark011     │ 2020-06-11 00:00:00 │         1 │          704 │ INR      │     1 │
│ Prod300      │ Cus021        │ Mark011     │ 2020-06-11 00:00:00 │         1 │          157 │ INR      │     1 │
│ Prod302      │ Cus021        │ Mark011     │ 2018-12-05 00:00:00 │         1 │          343 │ INR      │     1 │
│ Prod053      │ Cus022        │ Mark011     │ 2020-04-27 00:00:00 │         1 │          704 │ INR      │     1 │
│ Prod095      │ Cus022        │ Mark011     │ 2018-06-07 00:00:00 │         2 │

This helps detect hidden whitespace problems before cleaning.

In [124]:
con.sql(
    """
    SELECT
      markets_name,
      LENGTH(markets_name) AS name_length
    FROM markets
    ORDER BY markets_name;
    """
)

┌──────────────┬─────────────┐
│ markets_name │ name_length │
│   varchar    │    int64    │
├──────────────┼─────────────┤
│ Ahmedabad    │           9 │
│ Bengaluru    │           9 │
│ Bhopal       │           6 │
│ Bhopal       │           6 │
│ Bhubaneshwar │          12 │
│ Chennai      │           7 │
│ Delhi NCR    │           9 │
│ Hyderabad    │           9 │
│ Kanpur       │           6 │
│ Kochi        │           5 │
│ Lucknow      │           7 │
│ Mumbai       │           6 │
│ Nagpur       │           6 │
│ New York     │           8 │
│ Paris        │           5 │
│ Patna        │           5 │
│ Surat        │           5 │
├──────────────┴─────────────┤
│ 17 rows          2 columns │
└────────────────────────────┘

Core String Cleaning Functions

Add this as a Colab reference heading.

String Normalization Functions
TRIM()

Removes spaces from both sides.

TRIM('  Delhi  ')

Result:Delhi

UPPER()

Standardizes case.

UPPER('Delhi')

Result: DELHI

LOWER()

LOWER('DELHI')

Result: delhi

REPLACE()

Replace characters.

REPLACE('Delhi-NCR','-',' ')

Result: Delhi NCR

REGEXP_REPLACE()

Used for pattern cleaning.

Example remove numbers:

REGEXP_REPLACE('Delhi123','[0-9]','')

Result: Delhi

**Real Transformation Query (Using Your Dataset)**

Now we do a real cleaning pipeline.

Business case:

-Standardize market names

-Remove spaces

-Convert to uppercase

In [125]:
con.sql(
    """
    SELECT
      market_code,
      markets_name AS original_market,

      UPPER(TRIM(markets_name)) AS normalized_market
    FROM markets

    """
)

┌─────────────┬─────────────────┬───────────────────┐
│ market_code │ original_market │ normalized_market │
│   varchar   │     varchar     │      varchar      │
├─────────────┼─────────────────┼───────────────────┤
│ Mark001     │ Chennai         │ CHENNAI           │
│ Mark002     │ Mumbai          │ MUMBAI            │
│ Mark003     │ Ahmedabad       │ AHMEDABAD         │
│ Mark004     │ Delhi NCR       │ DELHI NCR         │
│ Mark005     │ Kanpur          │ KANPUR            │
│ Mark006     │ Bengaluru       │ BENGALURU         │
│ Mark007     │ Bhopal          │ BHOPAL            │
│ Mark008     │ Lucknow         │ LUCKNOW           │
│ Mark009     │ Patna           │ PATNA             │
│ Mark010     │ Kochi           │ KOCHI             │
│ Mark011     │ Nagpur          │ NAGPUR            │
│ Mark012     │ Surat           │ SURAT             │
│ Mark013     │ Bhopal          │ BHOPAL            │
│ Mark014     │ Hyderabad       │ HYDERABAD         │
│ Mark015     │ Bhubaneshwar

Now we apply REGEXP cleaning on the transactions table.

Business problem:

Some pipelines accidentally insert numeric characters into product codes.

Example dirty value:

Prod312#1

Prod312-Temp

Prod312_v2

We want to extract the real product code.

In [126]:
con.sql(
    """
    SELECT
      product_code,

      REGEXP_REPLACE(product_code, '[^A-Za-z0-9]', '') AS cleaned_product_code
    FROM transactions
    """
)

┌──────────────┬──────────────────────┐
│ product_code │ cleaned_product_code │
│   varchar    │       varchar        │
├──────────────┼──────────────────────┤
│ Prod001      │ Prod001              │
│ Prod001      │ Prod001              │
│ Prod002      │ Prod002              │
│ Prod002      │ Prod002              │
│ Prod002      │ Prod002              │
│ Prod003      │ Prod003              │
│ Prod003      │ Prod003              │
│ Prod003      │ Prod003              │
│ Prod003      │ Prod003              │
│ Prod003      │ Prod003              │
│    ·         │    ·                 │
│    ·         │    ·                 │
│    ·         │    ·                 │
│ Prod054      │ Prod054              │
│ Prod054      │ Prod054              │
│ Prod054      │ Prod054              │
│ Prod054      │ Prod054              │
│ Prod054      │ Prod054              │
│ Prod054      │ Prod054              │
│ Prod054      │ Prod054              │
│ Prod054      │ Prod054              │


**JSON Parsing Concept**

Modern data pipelines often ingest semi-structured data such as:

API responses

application events

clickstream logs

IoT telemetry

These datasets frequently store information in JSON columns.

Example JSON record:

{

  "device": "mobile",

  "products": ["Prod101","Prod205"],

  "location": {

      "city": "Delhi",

      "country": "India"
  }
}

**Step 1 — Create JSON Dataset**

In [127]:
con.sql(
    """
    CREATE TABLE events AS
    SELECT * FROM (
      VALUES
      (1, '2024-01-01', '{"device":"mobile","products":["Prod101","Prod205"],"location":{"city":"Delhi","country":"India"}}'),
      (2, '2024-01-02', '{"device":"desktop","products":["Prod305"],"location":{"city":"Mumbai","country":"India"}}'),
      (3, '2024-01-03', '{"device":"tablet","products":["Prod401","Prod402","Prod403"],"location":{"city":"Bangalore","country":"India"}}')
    ) AS t(event_id, event_date, event_payload);
    """
)

In [128]:
con.sql(
    """
    SELECT * FROM events;
    """
)

┌──────────┬────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ event_id │ event_date │                                                  event_payload                                                   │
│  int32   │  varchar   │                                                     varchar                                                      │
├──────────┼────────────┼──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│        1 │ 2024-01-01 │ {"device":"mobile","products":["Prod101","Prod205"],"location":{"city":"Delhi","country":"India"}}               │
│        2 │ 2024-01-02 │ {"device":"desktop","products":["Prod305"],"location":{"city":"Mumbai","country":"India"}}                       │
│        3 │ 2024-01-03 │ {"device":"tablet","products":["Prod401","Prod402","Prod403"],"location":{"city":"Bangalore","country":"India"}} │
└──────────┴─

**Step 2 — Extract JSON Fields**

In [129]:
con.sql(
    """
    SELECT
      event_id,

      json_extract_string(event_payload, '$.device') AS device,

      json_extract_string(event_payload, '$.location.city') AS city

    FROM events
    """
)

┌──────────┬─────────┬───────────┐
│ event_id │ device  │   city    │
│  int32   │ varchar │  varchar  │
├──────────┼─────────┼───────────┤
│        1 │ mobile  │ Delhi     │
│        2 │ desktop │ Mumbai    │
│        3 │ tablet  │ Bangalore │
└──────────┴─────────┴───────────┘

**Step 3 — Extract Array From JSON**

The products field is an array.

In [130]:
con.sql(
    """
    SELECT
      event_id,

      json_extract(event_payload, '$.products') AS products_array
    FROM events
    """
  )

┌──────────┬─────────────────────────────────┐
│ event_id │         products_array          │
│  int32   │              json               │
├──────────┼─────────────────────────────────┤
│        1 │ ["Prod101","Prod205"]           │
│        2 │ ["Prod305"]                     │
│        3 │ ["Prod401","Prod402","Prod403"] │
└──────────┴─────────────────────────────────┘

**Step 4 — Flatten Array Using UNNEST**

Now convert arrays into rows.

In [131]:
con.sql(
    """
    SELECT
      event_id,
      product
    FROM events,
    UNNEST(
      json_extract(event_payload, '$.products')::VARCHAR[]
    ) AS product;
    """
)

┌──────────┬────────────────────────┐
│ event_id │        product         │
│  int32   │ struct(unnest varchar) │
├──────────┼────────────────────────┤
│        1 │ {'unnest': Prod101}    │
│        1 │ {'unnest': Prod205}    │
│        2 │ {'unnest': Prod305}    │
│        3 │ {'unnest': Prod401}    │
│        3 │ {'unnest': Prod402}    │
│        3 │ {'unnest': Prod403}    │
└──────────┴────────────────────────┘